---
title: "Chapter 4: Machine Learning for Time Series"
jupyter:
  kernelspec:
    display_name: Python (timeSeries-course)
    language: python
    name: timeseries-course
editor_options: 
  chunk_output_type: console
---

# Machine Learning for Time Series

### Learning Objectives

By the end of this chapter, you will be able to:

1.  Transform time series data into supervised learning problems through feature engineering
2.  Create and select appropriate features including lags, rolling statistics, calendar features, and Fourier terms
3.  Apply traditional ML models (Ridge, Lasso, Random Forest, XGBoost, LightGBM) to time series forecasting
4.  Understand and implement the Prophet model for business forecasting
5.  Design and implement hybrid approaches combining statistical and ML methods
6.  Critically evaluate when ML approaches offer advantages over classical statistical methods

### Topics Covered

1.  From Time Series to Supervised Learning: The ML Paradigm Shift
2.  Feature Engineering for Time Series
3.  Traditional Machine Learning Models
4.  Prophet: Additive Regression Models
5.  Hybrid Approaches: Combining Statistical and ML Methods
6.  Model Selection and Comparison

------------------------------------------------------------------------

## 1. From Time Series to Supervised Learning: The ML Paradigm Shift

### 1.1 The Fundamental Transformation

Classical time series models such as ARIMA operate directly on the sequential structure of data, explicitly modeling temporal dependencies through autoregressive terms and moving averages. Machine learning approaches take a fundamentally different path: they transform the forecasting problem into a tabular supervised learning problem.

This transformation has important implications:

**Advantages:**

-   Access to the entire ML toolkit (regularization, ensemble methods, gradient boosting)
-   Natural handling of multiple exogenous variables
-   No stationarity requirements
-   Ability to capture complex non-linear relationships

**Challenges:**

-   Loss of explicit temporal structure awareness
-   Feature engineering becomes critical
-   Risk of temporal data leakage
-   More hyperparameters to tune

### 1.2 The Transformation Process

Consider a time series $\{y_1, y_2, \ldots, y_T\}$. To apply ML models, we create a design matrix $\mathbf{X}$ where each row represents an observation and columns contain features (predictors), with the target variable $\mathbf{y}$ being the value we want to forecast.

For a simple autoregressive structure with $p$ lags:

| Time $t$ | $X_1$ ($y_{t-1}$) | $X_2$ ($y_{t-2}$) | ... | $X_p$ ($y_{t-p}$) | Target ($y_t$) |
|------------|------------|------------|------------|------------|------------|
| $p+1$ | $y_p$ | $y_{p-1}$ | ... | $y_1$ | $y_{p+1}$ |
| $p+2$ | $y_{p+1}$ | $y_p$ | ... | $y_2$ | $y_{p+2}$ |
| ... | ... | ... | ... | ... | ... |
| $T$ | $y_{T-1}$ | $y_{T-2}$ | ... | $y_{T-p}$ | $y_T$ |

> **Critical Question:** What determines the "right" number of lags $p$ to include? How might this relate to concepts we learned in Chapter 3 (PACF analysis)?

::: {.callout-note collapse="true"}
In Chapter 3, we used the Partial Autocorrelation Function (PACF) to identify the order $p$ of an Autoregressive process. Recall the definition of an AR($p$) process:

$$Y_t = c + \phi_1 Y_{t-1} + \phi_2 Y_{t-2} + \dots + \phi_p Y_{t-p} + \varepsilon_t$$

This equation states that the value at time $t$ is a linear combination of the $p$ previous values.

How does this translate to Machine Learning? In the ML context, this equation is simply a Linear Regression where the "features" are the past values. Therefore, the order $p$ identified by the PACF tells us exactly how many lag features we should generate.

-   If PACF cuts off after lag 3: It means $Y_t$ depends on $Y_{t-1}, Y_{t-2}, Y_{t-3}$.
-   Feature Engineering Action: We must create columns `lag_1`, `lag_2`, and `lag_3` in our design matrix. Including `lag_4` would be redundant (adding noise), and excluding `lag_3` would mean missing signal (underfitting).
:::

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Import forecasting libraries
from skforecast.datasets import fetch_dataset
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

In [ ]:
#| label: load-data
#| echo: true

# Load example dataset: Australian drug sales (monthly)
data = fetch_dataset(name='h2o', raw=True)
data['fecha'] = pd.to_datetime(data['fecha'], format='%Y-%m-%d')
data = data.set_index('fecha')
data = data.asfreq('MS')
data = data.rename(columns={'x': 'sales'})

# Quick visualization
fig, ax = plt.subplots(figsize=(12, 5))
data['sales'].plot(ax=ax, color='steelblue', linewidth=1.5)
ax.set_title('Monthly Corticosteroid Drug Sales in Australia', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Sales ($AUD)')
plt.tight_layout()
plt.show()

print(f"Dataset shape: {data.shape}")
print(f"Date range: {data.index.min()} to {data.index.max()}")

### 1.3 Recursive vs Direct Forecasting Strategies

When forecasting multiple steps ahead, two main strategies exist:

**Recursive (Iterative) Strategy:**

-   Train a single model to predict one step ahead
-   Use predictions as inputs for subsequent forecasts
-   Errors can accumulate over the forecast horizon

```{mermaid}
%%| echo: false
%%| label: fig-recursive-strategy
%%| fig-cap: "Recursive (Iterative) Forecasting Strategy"

flowchart LR
    %% Input nodes
    Hist[Historical Data<br>X_t] --> M1(Model)
    
    %% Step 1
    M1 --> P1[Forecast<br>t+1]
    
    %% Feedback (Recursive Loop)
    P1 -.->|Input| M2(Model)
    Hist -.-> M2
    
    %% Step 2
    M2 --> P2[Forecast<br>t+2]
    
    %% Feedback 2
    P2 -.->|Input| M3(Model)
    P1 -.-> M3
    Hist -.-> M3
    
    %% Step 3
    M3 --> P3[Forecast<br>t+3]

    %% Styles (Optional: soft academic colors)
    style P1 fill:#e6f3ff,stroke:#333,stroke-width:1px
    style P2 fill:#fff0e6,stroke:#333,stroke-width:1px
    style P3 fill:#e6ffe6,stroke:#333,stroke-width:1px
    style M1 fill:#f0f0f0,stroke:#666
    style M2 fill:#f0f0f0,stroke:#666
    style M3 fill:#f0f0f0,stroke:#666
```

**Direct Strategy:**

-   Train separate models for each forecast horizon
-   Model $h$ predicts directly $y_{t+h}$ from features at time $t$
-   No error accumulation, but ignores relationships between horizons

```{mermaid}
%%| echo: false
%%| label: fig-direct-strategy
%%| fig-cap: "Direct Forecasting Strategy (Independent Models)"

flowchart TD
    %% Central Data Node
    Hist[Historical Data<br>X_t]
    
    %% Separate branches
    subgraph H1 [Horizon 1]
        direction LR
        M1(Model A) --> P1[Forecast<br>t+1]
    end
    
    subgraph H2 [Horizon 2]
        direction LR
        M2(Model B) --> P2[Forecast<br>t+2]
    end
    
    subgraph H3 [Horizon 3]
        direction LR
        M3(Model C) --> P3[Forecast<br>t+3]
    end

    %% Connections
    Hist --> M1
    Hist --> M2
    Hist --> M3

    %% Styles
    style M1 fill:#cce5ff,stroke:#004085
    style M2 fill:#d4edda,stroke:#155724
    style M3 fill:#f8d7da,stroke:#721c24
    
    style P1 fill:#fff,stroke:#333,stroke-dasharray: 5 5
    style P2 fill:#fff,stroke:#333,stroke-dasharray: 5 5
    style P3 fill:#fff,stroke:#333,stroke-dasharray: 5 5
```

> **Think About It:** Under what circumstances might recursive forecasting outperform direct forecasting, and vice versa?

::: {.callout-note collapse="true"}
Choosing between recursive and direct strategies involves a trade-off between computational cost, error propagation, and forecast coherence.

| Feature | Recursive (Iterative) | Direct (Independent) |
|:-----------------------|:-----------------------|:-----------------------|
| **Model Count** | **1 Model** (Low computational cost) | $h$ Models (High cost for long horizons) |
| **Error Behavior** | **Accumulates**: Errors in $t+1$ affect $t+2$. Forecast quality degrades rapidly over time. | **Constant**: Error at $t+h$ is independent of $t+1$. No "snowball effect". |
| **Dependencies** | **Preserved**: Produces smooth, coherent trajectories because $y_{t+2}$ depends on $y_{t+1}$. | **Ignored**: Can produce "jagged" or inconsistent forecasts because predictions are independent. |
| **Best For...** | Short-to-medium horizons, high-frequency data, or when the underlying dynamics are stable. | Long horizons, complex seasonality, or when specific future points depend on distant past features. |

**Guideline:** Start with the Recursive strategy (default in most libraries like `skforecast` or `darts`) due to its efficiency. Switch to Direct if you observe that accuracy drops sharply after the first few steps or if you are targeting a specific distant horizon.
:::

------------------------------------------------------------------------

## 2. Feature Engineering for Time Series

Feature engineering is arguably the most critical step in ML-based time series forecasting. The quality and relevance of features often matters more than the choice of model.

### 2.1 Lag Features

Lag features capture the autoregressive nature of time series. The value at time $t-k$ becomes a predictor (feature) for the value at time $t$ (target).

$$X^{(lag_p)}_t = y_{t-p}$$

**Considerations:**

-   **Number of lags:** Guided by ACF/PACF analysis (as seen in Chapter 3) or domain knowledge.
-   **Seasonal lags:** For monthly data with yearly seasonality, it is crucial to include lag 12 ($y_{t-12}$).
-   **Missing Values:** Creating lags inevitably introduces `NaN` values at the beginning of the series (we cannot look back before the start date). These rows must usually be dropped before training.

There are several ways to implement this transformation in Python. We will explore three common approaches, ranging from native Pandas to specialized ML libraries.

#### 2.1.1. Method 1: Using Pandas

For quick prototyping or simple models, we don't need external libraries. We can leverage Pandas' `.shift()` method combined with dictionary comprehension. This approach is highly efficient and keeps the codebase lightweight.

In [ ]:
#| label: lag-features-pandas
#| echo: true

lag_p = 12 #no. of lags we consider
data_pandas = data.assign(**{f'lag_{l}': data['sales'].shift(l) for l in list(range(1, lag_p+1))})
print("\nMethod 1: Native Pandas")
print(data_pandas.head(5))
print(f"Shape: {data_pandas.shape}")

The expression `{f'lag_{l}': data['sales'].shift(l) for l in lags_list}` creates a dictionary where keys are column names (e.g., `lag_1`) and values are the shifted series. The `**` operator unpacks this dictionary into the `.assign()` method, creating all columns simultaneously.

#### 2.1.2. Method 2: Using `skforecast`

The library `skforecast` previously used in the course has also this utility already implemented. The function `create_train_X_y` automatically generates the lag matrix ($X$) and separates it from the target ($y$). This is the method used internally by `ForecasterRecursive`.

In [ ]:
#| label: lag-features-skforecast
#| echo: true

from skforecast.recursive import ForecasterRecursive
from sklearn.linear_model import LinearRegression

# 1. We need to instantiate a Forecaster with a regression model 
# (LinearRegression in this case)
# This object knows how to handle the lags
forecaster = ForecasterRecursive(
    regressor=LinearRegression(),
    lags=12
)

# Ask the forecaster to create the matrices for us
X_lags, y_target = forecaster.create_train_X_y(y=data['sales'])

# Merge just for visualization
data_skforecast = pd.concat([y_target, X_lags], axis=1)

print("\nMethod 2: Skforecast Utility (via Forecaster)")
print(data_skforecast.head(5))

#### 2.1.3. Method 3: Using `feature_engine`

In production environments where we use `scikit-learn` pipeline objects, it is best to use transformers that fit into that ecosystem. The `feature_engine` library provides the `LagFeatures` transformer, which follows the standard fit/transform API.

In [ ]:
#| label: lag-features-feature-engine
#| echo: true

from feature_engine.timeseries.forecasting import LagFeatures

# 1. Define the transformer
transformer = LagFeatures(
    variables=['sales'],    # Column to lag
    periods=list(range(1, 13)), # Lags to create
    drop_original=False     # Keep the original sales column
)

# 2. Transform the data
data_feature_engine = transformer.fit_transform(data)
print("\nMethod 3: Feature Engine")
print(data_feature_engine.head(5))

> For the exercises we will primarily rely on Method 2 (skforecast) implicitly when training models, or Method 1 (Pandas) when we need to inspect the data manually.

### 2.2 Rolling Statistics

Rolling (or moving) statistics capture local patterns and trends by summarizing behavior over a sliding window of size $w$. Note that $w$ does not necesary correspond to the number of lags $p$. Unlike lag features which look at a specific points in time ($t-1,\ldots, t-p$), rolling features aggregate a range of past values (values within the window of size $w$) to describe the "state" of the series.

To avoid data leakage, features at time $t$ must only be calculated using information available up to $t-1$. Therefore, we apply the rolling window to the shifted series: $\{y_{t-1}, y_{t-2}, \dots, y_{t-w}\}$.

Common rolling features include:

-   **Rolling mean:**

    $$
    \bar{y}_{t,w} = \frac{1}{w}\sum_{i=1}^{w} y_{t-i}
    $$

-   **Rolling standard deviation:** Captures local volatility

    $$
    \sigma_{t,w} = \sqrt{\frac{1}{w-1} \sum_{i=1}^{w} \left(y_{t-i} - \bar{y}_{t,w}\right)^2}
    $$

-   **Rolling min/max:** Captures local extremes

    $$
    Min_{t,w} = \min \{ y_{t-i} \mid 1 \le i \le w \}, \ \ \ Max_{t,w} = \max \{ y_{t-i} \mid 1 \le i \le w \}
    $$

-   **Rolling quantiles:** More robust summary statistics being is less sensitive to outliers. In python, by default, it is estimated via linear interpolation of order statistics.

    Let $\{y_{(1)} \le y_{(2)} \le \dots \le y_{(w)}\}$ denote the order statistics of the window at time $t$. For a probability $q \in [0, 1]$, the rolling quantile is defined as: $$
        Q_{t,w}^{(q)} = y_{(\lfloor h \rfloor)} + (h - \lfloor h \rfloor) \left( y_{(\lceil h \rceil)} - y_{(\lfloor h \rfloor)} \right)$$

    where $h = (w-1)q + 1$ is the virtual index, and $\lfloor \cdot \rfloor$ and $\lceil \cdot \rceil$ are the floor and ceiling functions, respectively.

Below, we explore three methods to implement these features using a window size $w=6$. Each suitable for different stages of the modeling lifecycle.

#### 2.2.1. Method 1: Using Pandas

Pandas provides the `.rolling().agg()` pattern, which is efficient and flexible for data analysis and visualization.

**Pros:** No external dependencies; very fast for prototyping.

**Cons:** Static calculation. It does not support recursive forecasting automatically (you cannot easily update the window during prediction steps).

In [ ]:
#| label: rolling-features-pandas
#| echo: true

# usually we need to define the specific quantile we want to use
def q25(x): return x.quantile(0.25)
def q75(x): return x.quantile(0.75)

stats_funcs = ['mean', 'std', 'min', 'max', q25, q75]


# Shift the data once to avoid leakage
shifted_data = data['sales'].shift(1)
    
# Compute rolling window statistics
window_stats = shifted_data.rolling(window=6).agg(stats_funcs)
    
data_pandas = pd.concat([data_pandas,window_stats], axis=1)

print("Method 1: Pandas")
print(data_pandas.dropna().head(5))

#### 2.2.2. Method 2: Using `skforecast`

This is the recommended method for forecasting models. Instead of creating static columns, we define a `RollingFeatures` object that the forecaster uses.

**Pros:** Handles recursive forecasting. When predicting $t+2$, the model automatically calculates the rolling stats using the prediction of $t+1$.

**Cons:** Syntax is specific to the skforecast ecosystem. Additionally, unlike Pandas, `skforecast.RollingFeatures` is optimized for performance and does not support custom Python functions (like specific quantiles `q25`). It only supports a fixed set of compiled statistics to ensure speed during recursive forecasting:

|  |  |  |
|------------------------|------------------------|------------------------|
| **Statistic** | **String Argument** | **Description** |
| **Mean** | `'mean'` | Arithmetic mean of the values in the window. |
| **Standard Deviation** | `'std'` | Standard deviation (measure of volatility). |
| **Minimum** | `'min'` | Minimum value observed in the window. |
| **Maximum** | `'max'` | Maximum value observed in the window. |
| **Median** | `'median'` | The 50th percentile (robust central tendency). |
| **Sum** | `'sum'` | Total sum of values in the window. |
| **Ratio Min/Max** | `'ratio_min_max'` | Calculated as $min / max$. Useful for detecting range compression. |
| **Coef. of Variation** | `'coef_variation'` | Calculated as $std / mean$. Measures relative volatility. |

In [ ]:
#| label: skforecast-rolling-example
#| echo: true

import pandas as pd
from sklearn.linear_model import LinearRegression
from skforecast.recursive import ForecasterRecursive
from skforecast.preprocessing import RollingFeatures

# Configure the RollingFeatures object
# This object encapsulates the logic: "Calculate these stats over the past 6 months"
# NOTE: No manual shifting is required here; skforecast handles data alignment internally
# to prevent data leakage.
rolling_transformer = RollingFeatures(
    stats=['mean', 'std', 'min', 'max'], 
    window_sizes=6 
)

# Instantiate the Forecaster
# We combine traditional point lags (12 months) with window statistics (6 months)
forecaster = ForecasterRecursive(
    regressor=LinearRegression(),
    lags=12,                            # Point lags (t-1 ... t-12)
    window_features=rolling_transformer # Window aggregated features
)

# Ask the forecaster to create the matrices for us.
# The create_train_X_y method automatically:
# - Applies the lags
# - Calculates rolling features (respecting causal order)
# - Drops the initial rows containing NaNs
X, y = forecaster.create_train_X_y(y=data['sales'])

# Merge just for visualization
data_skforecast = pd.concat([y, X], axis=1)

print("\nMethod 2: Skforecast Utility (via Forecaster)")
print(data_skforecast.head(5))

#### 2.2.3. Method 3: Using `feature_engine`

For scikit-learn pipelines that require strict separation of steps, the `feature_engine` library offers the `WindowFeatures` transformer. This is ideal for production environments but requires stricter data preparation than Pandas.

**Pros:** It is defined according to scikit-learn API and directly integrates into the `sklearn` pipelines.

**Cons:** Strict input requirements (must be a DataFrame, NaNs not allowed, and data must be manually shifted to avoid leakage before transforming). Additionally, similarly to `skforecast`, it only accepts standard methods (`'mean'`, `'std'`, `'median'`, ...) and does not support custom functions like `q25` or `q75`.

In [ ]:
#| label: rolling-features-feature-engine
#| echo: true

from feature_engine.timeseries.forecasting import WindowFeatures
    
# 1. Define the transformer (only allowed statistics)
transformer = WindowFeatures(
  variables=['sales'],
  window=[6], 
  functions=['mean', 'std', 'min', 'max', 'median'],
  drop_original=False
)
    
# 2. Data Preparation (Crucial Step)
# feature_engine is strict:
#   a) The input must be a DataFrame (not a Series).
#   b) The input must NOT contain NaNs.
#   c) We must shift the data manually to avoid leakage before transforming.
    
shifted_df = data['sales'].shift(1).to_frame().dropna()
    
# 3. Transform
# The transformer applies the rolling window to the clean, shifted data
data_fe = transformer.fit_transform(shifted_df)
    
print("Method 3: Feature Engine (Standard stats only)")
print(data_fe.head())

Notice the line `shifted_df = data['sales'].shift(1).to_frame().dropna()`. Unlike `skforecast` (which handles alignment internally) or Pandas (which tolerates NaNs), `feature_engine` will raise a `ValueError` if passed any missing values. Therefore, we must explicitly clean the shifted data before feeding it into the transformer.

### 2.3 Calendar Features

Calendar features encode temporal patterns that repeat according to fixed human cycles. Unlike lag features (which depend on data history), these are deterministic (e.g. we know exactly when Christmas will fall in 2050 without needing a model to predict it).

Common calendar features include:

-   **Time of year:** Month (1-12), Quarter (1-4), Week of year (1-52).
-   **Time of week:** Day of week (0-6), "Is Weekend" binary flag.
-   **Special events:** Holidays, Fiscal year boundaries, ...

#### 2.3.1. Method 1: Pandas & Holidays Lib

Pandas makes accessing datetime properties intuitive via the `.dt` accessor. For handling national holidays (which often change dates, like Easter), the `holidays` library is recomended over writing manual rules.

**Pros:** Maximum flexibility for domain logic (e.g., defining "Summer" based on location) and complex holiday handling.

**Cons:** Requires more manual coding; not automatically wrapped in a Scikit-learn transformer.

In [ ]:
#| label: calendar-features-pandas
#| echo: true

import holidays

# 1. Create a specific DataFrame for this analysis
data_calendar = data.copy()

# 2. Basic Calendar Components
data_calendar['year'] = data_calendar.index.year
data_calendar['month'] = data_calendar.index.month
data_calendar['quarter'] = data_calendar.index.quarter

# 3. Automatic Holidays (using 'holidays' library)
# We initialize the calendar for Australia (State of Victoria)
# This automatically calculates movable feasts (Easter, etc.)
au_holidays = holidays.AUS(subdiv='VIC')
# holidays uses ISO 3166-2 codes for countries and states/provices (subdivisions)
# see https://www.iso.org/obp/ui/es/ for codes
# print(holidays.AUS.subdivisions)
# but the name of the country is also ususally supported
# print(holidays.Australia.subdivisions)

# Create a binary feature: 1 if date is holiday, 0 otherwise
data_calendar['is_holiday'] = data_calendar.index.map(lambda x: 1 if x in au_holidays else 0)

# 4. Domain-Specific Logic (Seasonality)
# Australia (Southern Hemisphere) -> Summer: Dec, Jan, Feb | Winter: Jun, Jul, Aug
data_calendar['is_summer'] = data_calendar['month'].isin([12, 1, 2]).astype(int)

print("Method 1: Pandas with Holidays (Australia Context)")
print(data_calendar[['sales', 'month', 'is_summer', 'is_holiday']].sample(5).sort_index())

> Notice the `is_summer` correction we use the summer months from Australia. For countries in the north hemisphere we should adapt the values otherwise our model would look for peak demand in the coldest months.

#### 2.3.2. Method 2: Skforecast (Using `exog`)

*`skforecast` don't have a `CalendarFeatures` class like it does for `RollingFeatures`*. Unlike rolling statistics (which depend on previous predictions via recursion), calendar features are known in advance (deterministic). Therefore, in `skforecast`, they are treated as exogenous variables.

1.  We compute them using Pandas (Method A).

2.  We pass them to the `exog` argument.

In [ ]:
#| label: calendar-features-skforecast
#| echo: true

from skforecast.recursive import ForecasterRecursive
from sklearn.linear_model import LinearRegression

# 1. Prepare data
# We use the DataFrame created in Method A
y = data_calendar['sales']
exog = data_calendar[['month', 'quarter', 'is_summer']]

# 2. Instantiate Forecaster
forecaster = ForecasterRecursive(
    regressor=LinearRegression(),
    lags=12
)

# 3. Fit with Exogenous variables
# The model learns: Sales = f(Lags, Month, Quarter, Is_Summer)
forecaster.fit(y=y, exog=exog)

# Ask the forecaster to create the matrices for us.
X, y = forecaster.create_train_X_y(y=data['sales'], exog=exog)

# Merge just for visualization
data_skforecast = pd.concat([y, X], axis=1)
data_skforecast.head(5)

> **Important:** When calling `predict()`, you must provide the `exog` values for the future steps. Since calendar features are deterministic, you can easily generate a DataFrame with the future months/quarters to feed the model.

In [ ]:
#| label: skforecast-predict-context
#| echo: true

import pandas as pd

# 1. Define the prediction horizon
steps = 2

# 2. Generate the future timeline
# We create dates starting immediately after the last observation in our data
last_date = data_calendar.index[-1]
future_dates = pd.date_range(
    start=last_date + pd.DateOffset(months=1), 
    periods=steps, 
    freq='MS' # 'MS' = Month Start frequency
)

# 3. Create Future Exogenous Variables
# We must provide the model with the calendar data for these future dates.
# BEST PRACTICE: Reuse the code from Method A to ensure 
# consistency (same holiday logic, same season definitions).
future_df = pd.DataFrame(index=future_dates)
future_df['year'] = future_df.index.year
future_df['month'] = future_df.index.month
future_df['quarter'] = future_df.index.quarter
future_df['is_holiday'] = future_df.index.map(lambda x: 1 if x in au_holidays else 0)
future_df['is_summer'] = future_df['month'].isin([12, 1, 2]).astype(int)

# Filter: Keep only the columns we used for training
future_exog = future_df[['month', 'quarter', 'is_summer']]

# 4. Predict
# We pass the prepared future exogenous variables to the predict method
pred_series = forecaster.predict(steps=steps, exog=future_exog)

# 5. Create a Summary DataFrame
# Concatenate the prediction with the features to see the full context
results_complete = pd.concat([pred_series, future_exog], axis=1)

print(f"Prediction for the next {steps} months with context:")
print(results_complete)

#### 2.3.3. Method 3: Using `feature_engine`

For Scikit-learn pipelines, manually defining functions is repetitive. `feature_engine` provides the `DatetimeFeatures` transformer to extract these automatically.

**Pros:** Integrates into `sklearn.pipeline.Pipeline`. Works directly on the DataFrame index or columns.

**Cons:** Less flexibility for custom domain logic (like the "Southern Hemisphere Summer" definition) compared to a custom Pandas function.

In [ ]:
#| label: calendar-features-pipeline
#| echo: true

from feature_engine.datetime import DatetimeFeatures
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
import holidays

# 1. Standard Calendar Features (using feature_engine)
dt_transformer = DatetimeFeatures(
    variables='index',
    features_to_extract=['month', 'quarter', 'year'],
    drop_original=False
)

# 2. Custom Holiday Logic (wrapped for Pipeline)
def add_holidays_column(df):
    df_copy = df.copy()
    au_holidays = holidays.Australia(state='VIC')
    # Logic to create the binary column
    df_copy['is_holiday'] = df_copy.index.map(lambda x: 1 if x in au_holidays else 0)
    return df_copy

# We wrap our function to make it compatible with scikit-learn
holiday_transformer = FunctionTransformer(add_holidays_column)

# 3. Create the Pipeline
# This pipeline now handles both standard extraction and custom holidays
calendar_pipeline = Pipeline([
    ('standard_dates', dt_transformer),
    ('custom_holidays', holiday_transformer)
])

# 4. Transform
data_pipeline_output = calendar_pipeline.fit_transform(data['sales'].to_frame().dropna())

print("\nMethod 3: Pipeline (Feature Engine + Custom Logic)")
print(data_pipeline_output.sample(5).sort_index())

#### 2.3.4. Advanced: Cyclical Encoding (The "Clock" Problem)

When we represent months ($1, 2, \dots, 12$), week days ($1, \dots, 7$), etc., as integers, linear models perceive a large "distance" between extremes (e.g., December -$12$- and January -$1$-) when they are really adjacent.To fix this topological problem, we can map these variables onto a circle using sine and cosine transformations. This ensures, for instance, that Month 12 and Month 1 are numerically close.Mathematically, this transformation creates a sine wave and a cosine wave with a period of 12. This is identical to a Fourier Series of Order $K=1$.

Note that our goal here is to encode the calendar variable (month, day, hour, ...) correctly to fix the distance. However, a similar approach can be used to model the response variable by means of a Harmonic Regression that integrates Fourier harmonics as features. This approach is introduced in the next section.

In [ ]:
#| label: cyclical-features
#| echo: true

from feature_engine.creation import CyclicalFeatures
import matplotlib.pyplot as plt

# 1. Create base calendar feature
data_cyclic = data.copy()
data_cyclic['month'] = data_cyclic.index.month

# 2. Apply Cyclical Encoding
# This generates 'month_sin' and 'month_cos'
cyclical = CyclicalFeatures(variables=['month'], drop_original=False)
data_cyclic = cyclical.fit_transform(data_cyclic)

print("Cyclical Encoding (Fixing the Dec-Jan distance):")
# Filter to show only relevant columns
print(data_cyclic[['month', 'month_sin', 'month_cos']].head(3))

# Visualization: Proof of the "Circle"
# Plotting sine vs cosine reveals the clock-like structure
plt.figure(figsize=(4, 4))
plt.scatter(data_cyclic['month_sin'], data_cyclic['month_cos'], c=data_cyclic['month'], cmap='viridis')
plt.title("Cyclical Mapping (The Clock)")
plt.xlabel("Sine")
plt.ylabel("Cosine")
plt.grid(True, linestyle='--', alpha=0.6)
plt.axhline(0, color='grey', lw=0.5)
plt.axvline(0, color='grey', lw=0.5)

# Add labels for Jan and Dec to prove they are close
plt.text(data_cyclic['month_sin'].iloc[0], data_cyclic['month_cos'].iloc[0], '  Jan', fontsize=9, va='center')
plt.text(data_cyclic['month_sin'].iloc[11], data_cyclic['month_cos'].iloc[11], '  Dec', fontsize=9, va='center')

plt.tight_layout()
plt.show();

### 2.4 Fourier Terms for Seasonality (Harmonic Regression)

In the previous section, we saw how Cyclical Encoding converts the "Month" variable into sine and cosine waves. This allows a linear model to fit a simple, smooth annual pattern (equivalent to a Fourier series of Order $K=1$).

However, if the seasonal pattern is more complex, a single smooth wave ($K=1$) cannot capture this shape and it would result in a high-biased model. To solve this, higher frequencies ($K=2, 3, \dots$) harmonics can be used in the Fourier decomposition. By combining these waves, the model can approximate any periodic shape, no matter how jagged or complex.

Thus, for a time series with a seasonal period $P$ (e.g., $P=12$ for annual seasonality in monthly data), we generate $K$ pairs of sine and cosine terms, known as harmonics. The regressors for the $k$-th harmonic at time $t$ are given by:

$$
X_{k,t}^{(sin)} = \sin\left(\frac{2\pi \cdot k \cdot t}{P}\right), \quad X_{k,t}^{(cos)} = \cos\left(\frac{2\pi \cdot k \cdot t}{P}\right)
$$

Where:

-   $k = 1, \dots, K$ represents the harmonic order.

-   $P$ is the length of the cycle.

-   $K$ is a hyperparameter controlling the model complexity ("spectral resolution").

The parameter $K$ determines the "spectral resolution" of the seasonality:

-   **Low** $K$ (e.g., $K=1$): Captures only the fundamental frequency. The model assumes a pure, smooth sinusoidal pattern.

-   **High** $K$: Allows the model to capture higher-frequency changes and sharper peaks.

-   **Maximum** $K$ ($P/2$): Known as the Nyquist limit. At this point, the Fourier terms can perfectly reproduce *any* periodic variation of length $P$, equivalent to using dummy variables but with a spectral representation.

**Harmonic Regression vs. Spectral Analysis**

It is crucial to distinguish this feature engineering technique from signal processing (spectral analysis - Fourier transformation). In harmonic regression:

-   We do not transform the target variable $Y$ into the frequency domain. The target remains in the time domain.

-   Instead, we construct synthetic features ($X$) consisting of unweighted sine and cosine waves.

-   The machine learning algorithm (e.g., Linear Regression) then learns the optimal weights (coefficients) for these waves to minimize the error against the target $Y$.

Mathematically, we model the time series as a linear combination of basis functions:

$$
Y_t = \beta_0 + \sum_{k=1}^{K} \left[ \beta_{k}^{(sin)} \sin\left(\frac{2\pi kt}{P}\right) + \beta_{k}^{(cos)} \cos\left(\frac{2\pi kt}{P}\right) \right] + \epsilon_t
$$

Where:

-   $Y_t$: The target variable.

-   Sine/Cosine terms: The Fourier term generated as features (bounded between -1 and 1).

-   $\beta$: The coefficients learned by the model. If a frequency is irrelevant, the model may learn $\beta \approx 0$.

-   $K$: spectral resolution of the model as seen above.

#### 2.4.1. Method 1: NumPy

We calculate the terms analytically based on the time index $t$. This provides precise control over the number of harmonics ($K$).

In [ ]:
#| label: fourier-features
#| echo: true

import numpy as np
import matplotlib.pyplot as plt

def create_fourier_features(df, period, order):
    """
    Generates Fourier basis functions (X variables) for a given periodicity.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input data with a DatetimeIndex.
    period : int
        The fundamental period of the cycle (P).
    order : int
        The number of harmonic pairs (K) to generate.
    """
    result = df.copy()
    
    # We use an integer sequence for 't' to represent the time step
    t = np.arange(len(result))
    
    for k in range(1, order + 1):
        # Angular frequency omega
        omega = 2 * np.pi * k / period
        
        # We generate the unweighted waves. 
        # The model will later assign weights to these columns.
        result[f'sin_{k}'] = np.sin(omega * t)
        result[f'cos_{k}'] = np.cos(omega * t)
    
    return result

# Application: Yearly seasonality (P=12) with first 2 harmonics (K=2)
order = 6
data_fourier = create_fourier_features(data, period=12, order=order)

print("Fourier Basis Functions (Inputs X):")
print(data_fourier.head(3))

# --- Visualization of the Basis Functions ---
fig, axes = plt.subplots(order, 2, figsize=(10, 6), sharex=True)

for k in range(1, order + 1):
    # Use _ = to suppress matplotlib text output
    _ = axes[k-1, 0].plot(data_fourier.index[:24], data_fourier[f'sin_{k}'][:24], label=f'k={k}')
    _ = axes[k-1, 0].set_title(f'{k}-th Harmonic Sin')
    _ = axes[k-1, 0].legend(loc='upper right', fontsize='small')
    
    _ = axes[k-1, 1].plot(data_fourier.index[:24], data_fourier[f'cos_{k}'][:24], label=f'k={k}', color='orange')
    _ = axes[k-1, 1].set_title(f'{k}-th Harmonic Cos')
    _ = axes[k-1, 1].legend(loc='upper right', fontsize='small')

plt.tight_layout()
plt.show()

#### 2.4.2. Method 2: Skforecast Integration

In `skforecast`, Fourier terms are strictly treated as deterministic exogenous variables. Since the sine and cosine values depend solely on the time index $t$ and the fixed period $P$, they are known indefinitely into the future.

We pass these pre-computed features to the `exog` parameter. The regressor will then estimate the coefficients during the `fit` process.

In [ ]:
#| label: fourier-skforecast
#| echo: true

from skforecast.recursive import ForecasterRecursive
from sklearn.linear_model import LinearRegression

# 1. Feature Selection
# We utilize the orthogonal basis functions generated in Method A
y = data_fourier['sales']
# Select K=2 harmonics (4 features total)
exog_fourier = data_fourier[['sin_1', 'cos_1', 'sin_2', 'cos_2']]

# 2. Model Initialization
forecaster = ForecasterRecursive(
    regressor=LinearRegression(),
    lags=12
)

# 3. Training
# The model learns: Y = Intercept + (Weights_Lags * Lags) + (Weights_Fourier * Fourier)
forecaster.fit(y=y, exog=exog_fourier)

# Ask the forecaster to create the matrices for us.
X, y = forecaster.create_train_X_y(y=data['sales'], exog=exog)

# Merge just for visualization
data_skforecast = pd.concat([y, X], axis=1)
data_skforecast.head(5)

#### 2.4.3. Method 3: Feature-engine

While feature_engine provides CyclicalFeatures for the fundamental frequency ($K=1$), it does not natively support higher-order Fourier series ($K > 1$).

To integrate complex seasonality into a Scikit-learn pipeline (ensuring reproducibility in production), we wrap our NumPy generation logic into a FunctionTransformer. This allows us to treat Fourier feature generation as just another step in our preprocessing pipeline.

In [ ]:
#| label: fourier-custom-transformer
#| echo: true

from sklearn.preprocessing import FunctionTransformer
import numpy as np

# 1. Define the Custom Fourier Logic
def add_fourier_terms(df, period=12, order=1):
    """
    Custom function to add Fourier terms.
    Compatible with Scikit-learn via kwargs.
    """
    df_new = df.copy()
    
    # We use the time index (0, 1, 2...) for calculation
    # In production, ensure 't' is calculated relative to a fixed epoch
    t = np.arange(len(df_new))
    
    for k in range(1, order + 1):
        omega = 2 * np.pi * k / period
        df_new[f'sin_{k}'] = np.sin(omega * t)
        df_new[f'cos_{k}'] = np.cos(omega * t)
        
    return df_new

# 2. Create the Scikit-learn Transformer
# We configure it for Yearly Seasonality with K=2 harmonics
fourier_transformer = FunctionTransformer(
    func=add_fourier_terms,
    kw_args={'period': 12, 'order': 2}, 
    validate=False # Allows DataFrames to pass through without check
)

# 3. Transform
# Now we can use .fit_transform() just like any other sklearn object, 
# or alternatively include it into a pipeline:
# Pipeline([('fourier', fourier_transformer), ...])
data_fourier_sklearn = fourier_transformer.fit_transform(data)

print("Method 3: Custom Sklearn Transformer (K=2)")
print(data_fourier_sklearn.filter(like='sin').head())

### 2.5 Complete Feature Engineering Pipeline

Up to this point, we have explored individual techniques to extract information from time series data. However, powerful Machine Learning models (like XGBoost, or Random Forests) rarely rely on a single type of feature. They thrive when fed a rich combination of signals.

In this section, we consolidate all the previous methods into a unified workflow. This function will act as our "Feature Factory," transforming a univariate time series (just a target column) into a multivariate Feature Matrix ($X$).

The Composition of the Feature Matrix:

Our pipeline will generate a dataset containing two distinct types of information:

1.  **Auto-regressive Features (Dynamic):** Lags and Rolling Window statistics. These capture the *recent history* and *momentum* of the series.

2.  **Deterministic Features (Static/Known):** Calendar and Fourier terms. These capture *fixed cycles* and *seasonality* that we know in advance.

$$
Y_{t} = f(\underbrace{Y_{t-1}, Y_{t-12}}_{\text{Lags}}, \underbrace{\mu_{t-1..t-6}}_{\text{Rolling}}, \underbrace{\text{Month}_t}_{\text{Calendar}}, \underbrace{\sin(\frac{2\pi t}{12})}_{\text{Fourier}})
$$

**A Note on Data Availability ("The Burn-in Period")**

Applying these transformations comes at a cost: Data Truncation. For instance:

-   Creating a `lag_12` requires the first 12 data points to look back.

-   Calculating a `rolling_mean_6` requires 6 prior data points.

-   Consequently, the beginning of our dataset will be filled with `NaN` (Not a Number) values. These rows must be dropped before training, effectively shortening our training set. This lost period is often called the "burn-in" period.

In [ ]:
#| label: feature-pipeline
#| echo: true

def create_ml_features(df, target_col, 
                       lags=None, 
                       rolling_windows=None, 
                       rolling_funcs=['mean', 'std'],
                       fourier_period=None, 
                       fourier_terms=None,
                       include_calendar=True):
    """
    Complete feature engineering pipeline for ML-based time series forecasting.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with DatetimeIndex and target column
    target_col : str
        Name of target variable
    lags : list of int, optional
        Lag values to create
    rolling_windows : list of int, optional
        Window sizes for rolling statistics
    rolling_funcs : list of str
        Aggregation functions for rolling windows
    fourier_period : int, optional
        Period for Fourier terms
    fourier_terms : int, optional
        Number of Fourier term pairs
    include_calendar : bool
        Whether to include calendar features
        
    Returns
    -------
    pd.DataFrame
        DataFrame with all features
    """
    result = df.copy()
    
    # 1. Lag features
    if lags is not None:
        for lag in lags:
            result[f'lag_{lag}'] = result[target_col].shift(lag)
    
    # 2. Rolling features
    if rolling_windows is not None:
        # We shift by 1 to ensure no leakage (using past data only)
        shifted_target = result[target_col].shift(1)
        for window in rolling_windows:
            for func in rolling_funcs:
                col_name = f'rolling_{func}_{window}'
                if func == 'mean':
                    result[col_name] = shifted_target.rolling(window).mean()
                elif func == 'std':
                    result[col_name] = shifted_target.rolling(window).std()
                elif func == 'min':
                    result[col_name] = shifted_target.rolling(window).min()
                elif func == 'max':
                    result[col_name] = shifted_target.rolling(window).max()
    
    # 3. Calendar features
    if include_calendar:
        result['month'] = result.index.month
        result['quarter'] = result.index.quarter
        result['year'] = result.index.year
    
    # 4. Fourier features
    if fourier_period is not None and fourier_terms is not None:
        t = np.arange(len(result))
        for k in range(1, fourier_terms + 1):
            omega = 2 * np.pi * k / fourier_period
            result[f'sin_{k}'] = np.sin(omega * t)
            result[f'cos_{k}'] = np.cos(omega * t)
    
    return result

# Apply complete feature engineering
df_features = create_ml_features(
    data, 
    target_col='sales',
    lags=[1, 2, 3, 6, 12],           # Include seasonal lag
    rolling_windows=[3, 6, 12],      # Multiple windows
    rolling_funcs=['mean', 'std'],
    fourier_period=12,               # Yearly seasonality
    fourier_terms=2,                 # 2 pairs of sin/cos
    include_calendar=True
)

# Drop rows with NaN (due to lags and rolling windows)
df_features_clean = df_features.dropna()

print(f"Original observations: {len(data)}")
print(f"Observations after feature engineering: {len(df_features_clean)}")
print(f"Features created: {len(df_features_clean.columns) - 1}") # -1 for target
print(f"\nFirst 5 Feature columns:\n{list(df_features_clean.columns)[:5]} ...")

#### **Exercise 4.1**

Use `feature_engine` or `scikit-learn` pipelines to obtain the same set of features obtained in the previous code. *(Hint: You will need to combine `LagFeatures`, `WindowFeatures`, `DatetimeFeatures` and a `FunctionTransformer` for the Fourier terms into a single Pipeline).*

### 2.6 Feature Redundancy and Model Stability

After generating the feature matrix, it is mandatory to perform a diagnostic check before modeling. The goal here is not necessarily to discard features immediately, but to understand the structure of our data and how it will interact with our chosen model family. While generating a comprehensive feature matrix is a necessary step, the indiscriminate use of all generated variables is rarely optimal.

In time series feature engineering, we face a fundamental violation of the i.i.d. assumption (Independent and Identically Distributed) common in traditional Machine Learning:

1.  **Multicollinearity** (dependence between predictors): Features like `lag_1` and `rolling_mean_3` are derived from the same signal, leading to high correlation between columns.

2.  **Autocorrelation** (dependence between cases): The sliding window approach creates overlapping samples. A row at time $t$ shares almost the same information as the row at $t+1$, meaning the "cases" are not independent.

#### 2.6.1. Implications for Modeling

The decision of "how to handle" this structure depends entirely on the algorithm and the objective:

-   **Linear Models (Linear Regression):**

    -   High collinearity and lack of independence between cases make the calculation of coefficients unstable (high variance).

    -   The Loss of Explainability: While we can still use these models for prediction, we lose the ability to perform classical statistical inference. The coefficients $\beta$ may be unbiased, but their variance is incorrectly estimated, making $p$-values and individual feature interpretations unreliable.

    -   We typically rely on regularization (Lasso or Ridge regression). These techniques mathematically penalize redundant coefficients to stabilize the model.

-   **Tree-Based Models (XGBoost, Random Forest):**

    -   These models are robust to collinearity regarding predictive performance.

    -   However, correlation affects interpretability. If two features are redundant, the tree might arbitrarily choose one for splitting, diluting "Feature Importance" metrics.

-   **Parsimony, Overfitting, and Validation:**

    -   Following Occam's Razor principle, simpler models are preferable. A high ratio of predictors ($p$) to observations ($N$) increases the risk of the model "memorizing" noise instead of learning patterns.

    -   The Role of Backtesting: Since our data is not independent, standard cross-validation is forbidden. To safely use ML models, we must establish a strict temporal validation scheme (Backtesting) to ensure that our predictive power is real and not an artifact of data leakage.

#### 2.6.2. Visual Inspection: The Correlation Matrix

We use a correlation matrix to identify these relationships. *For continuous variables (lags, rolling), we use Pearson correlation. For discrete/ordinal variables (like Month), Spearman rank correlation is often more appropriate, though Pearson acts as a good approximation for a quick scan with ordinal variables*

A correlation heatmap is the standard diagnostic tool to identify redundancy between continuous variables before training (used also as an appoximation when ordinal variables are present in the problem). We aim to identify clusters of features with correlation coefficients approaching $\pm 1$.

In [ ]:
#| label: feature-correlation
#| echo: true

import seaborn as sns
import matplotlib.pyplot as plt

# Apply complete feature engineering
df_features = create_ml_features(
    data, 
    target_col='sales',
    lags=[1, 2, 3, 6, 12],           # Include seasonal lag
    rolling_windows=[3, 6, 12],      # Multiple windows
    rolling_funcs=['mean', 'std'],
    fourier_period=12,               # Yearly seasonality
    fourier_terms=2,                 # 2 pairs of sin/cos
    include_calendar=True
)

# Drop rows with NaN (due to lags and rolling windows)
df_features_clean = df_features.dropna()

# 1. Compute Correlation Matrix
# We select strictly the numeric engineered features for inspection
cols_to_check = [c for c in df_features_clean.columns if 'lag' in c or 'rolling' in c]
corr_matrix = df_features_clean[cols_to_check].corr()

# 2. Plot Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, 
            annot=True, 
            fmt=".2f", 
            cmap='RdBu_r', # Red-Blue diverging colormap 
            center=0,
            cbar_kws={'label': 'Pearson Correlation Coefficient'})

plt.title("Correlation Matrix: Autoregressive vs. Rolling Features")
plt.tight_layout()
plt.show()

##### **Exercise 4.2**

Analyze the correlation matrix generated above:

1.  Structural Redundancy: Observe the relationship between `lag_1` and `rolling_mean_3`. Explain mathematically why their correlation is so high.

2.  Impact on Estimation: If you were to train a standard OLS (Ordinary Least Squares) Linear Regression with this feature set, how would this multicollinearity affect the variance of the estimated coefficients $\beta$?

## 3. Traditional Machine Learning Models

Once the time series has been transformed into a feature matrix using the techniques presented in Section 2, we can transition from classical forecasting to supervised learning. Unlike the statistical methods estudied in previuous chapters (e.g. ARIMA), these algorithms do not assume a specific temporal structure; instead, they attempt to learn a mapping function between the input features $X$ (lags, windows, calendar) and the target $y$.

As established in Section 2.6, these models are applied in a context where the standard i.i.d. assumption is violated. Therefore, we prioritize predictive performance through robust validation (Backtesting) over the statistical interpretability of individual coefficients.

### 3.1. Linear Models

The Multiple Linear Regression model attempts to predict the target as a weighted sum of the features:

$$
Y_t = \beta_0 + \beta_1 X_{1,t} + \beta_2 X_{2,t} + \dots + \beta_p X_{p,t} + \epsilon_t
$$

where:

-   $Y_t$ (Dependent Variable): The target or objective value we wish to forecast at time $t$.

-   $\beta_0$ (Intercept / Bias): The value predicted by the model if all features $X$ were zero. It represents the baseline level of the series.

-   $X_{i,t}, \dots, X_{p,t}$ (Features / Predictors): The input variables extracted during the feature engineering phase. In our context, these are the lags ($y_{t-1}, y_{t-k}$), rolling statistics, or calendar indicators, ...

-   $\beta_1, \dots, \beta_p$ (Coefficients / Weights): These are the parameters that the model "learns" during training. Each coefficient represents the change in $Y_t$ for a one-unit change in the corresponding feature $X$, assuming all other features remain constant.

-   $\epsilon_t$ (Error Term / Innovation): Also known as "noise," this term captures the difference between the model's prediction and the actual observed value. It represents the part of the data that cannot be explained by the features (random fluctuations or unobserved factors).

The standard approach to estimating the coefficients ($\beta$) is Ordinary Least Squares (OLS). This method works by finding the set of weights that minimizes the loss function known as the Sum of Squared Errors (SSE):

$$
J_{OLS}(\beta) = SSE = \sum_{t=1}^{n} (y_t - \hat{y}_t)^2
$$

By minimizing this function, OLS ensures that the resulting model has the smallest possible distance (in squared terms) between the actual observations and the predicted values.

However, in time series forecasting, OLS is often problematic. As established in Section 2.6, the high multicollinearity between predictive variables makes the calculation of the coefficients unstable. This leads to a model with high variance, where small changes in the training data can result in wildly different coefficient estimates, ultimately degrading predictive performance.To address these issues, we move beyond OLS and use Regularized Linear Models, which add a penalty to the loss function to stabilize the estimates:

-   **Ridge Regression (L2 Regularization):** Ridge regression addresses multicollinearity by adding a penalty proportional to the square of the magnitude of the coefficients. It is highly effective at handling multicollinearity by shrinking correlated coefficients toward each other rather than discarding them.

    $$
    J_{Ridge}(\beta) = \sum_{t=1}^{n} (y_t - \hat{y}_t)^2 + \lambda \sum_{j=1}^{p} \beta_j^2
    $$

    Where $\lambda$ is the regularization parameter. As $\lambda$ increases, the coefficients shrink toward zero (but never reach it), effectively distributing the weight among correlated features and reducing model variance.

-   **Lasso Regression (L1 Regularization):** Lasso (Least Absolute Shrinkage and Selection Operator) regression adds a penalty proportional to the absolute value of the coefficients:

    $$
    J_{Lasso}(\beta) = \sum_{t=1}^{n} (y_t - \hat{y}_t)^2 + \lambda \sum_{j=1}^{p} |\beta_j|
    $$

    Similarly to Ridge, the regularization also depends on the regularization parameter $\lambda$ so that large $\lambda$ values impose stronger penalization and lead $\beta$ values to zero. However, by contrast to Ridge, Lasso regression can shrink some coefficients exactly to zero. This provides automatic feature selection, effectively removing irrelevant or redundant predictors from the model.

Note that, since the regularization penalty (in both Ridge and Lasso) acts directly on the magnitude of the coefficients ($\beta$), all features must be on the same scale. Therefore, variables must be standardized before using regularization. Without scaling, a variable with a very small range would naturally require a much larger $\beta$ than a variable with a large range. Consequently, the model would penalize the smaller-scale variable more heavily simply because of its units, regardless of its actual predictive power. Standardization ensures that the penalty is applied "fairly" across all predictors.

#### 3.1.1. Implementation: Ridge vs. Lasso

The following example demonstrates how to implement these regularized models using `skforecast`. In this baseline version, we focus exclusively on the autoregressive part (using the last 24 months as predictors) to observe how regularization handles the high correlation between consecutive lags.

For linear models, feature scaling (standardization) is mandatory. We use a `Pipeline` to ensure that scaling is performed correctly during the training and forecasting process, preventing data leakage. Additionally, $\lambda = 0.1$ is assumed in this example (note that $\lambda$ parameter is denoted as `alpha` - $\alpha$ - in the `skforecast` implementation).

In [ ]:
#| label: linear-regularization-skforecast
#| echo: true

from skforecast.recursive import ForecasterRecursive
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt

# 1. Initialize Regressors with a Pipeline
# We include StandardScaler to ensure all features have mean=0 and std=1
ridge_regressor = make_pipeline(StandardScaler(), Ridge(alpha=0.1))
lasso_regressor = make_pipeline(StandardScaler(), Lasso(alpha=0.1))

# 2. Create Forecasters
# lags=24 means we use the last 2 years of data as predictors
forecaster_ridge = ForecasterRecursive(regressor=ridge_regressor, lags=24)
forecaster_lasso = ForecasterRecursive(regressor=lasso_regressor, lags=24)

# 3. Fit models
# y_train should be a pandas Series with a DatetimeIndex or PeriodIndex
forecaster_ridge.fit(y=data['sales'])
forecaster_lasso.fit(y=data['sales'])

# 4. Predict
steps = 24
pred_ridge = forecaster_ridge.predict(steps=steps)
pred_lasso = forecaster_lasso.predict(steps=steps)

# 6. Visualization
fig, ax = plt.subplots(figsize=(12, 6))
data.tail(50).plot(ax=ax, label='Training Data (Tail)', color='gray', alpha=0.5)
pred_ridge.plot(ax=ax, label='Ridge Forecast (L2)', color='blue', linestyle='--')
pred_lasso.plot(ax=ax, label='Lasso Forecast (L1)', color='red', linestyle='-.')

ax.set_title("Sales Forecast: Ridge vs. Lasso Regularization")
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 7. Inspect Coefficients
lasso_coefs = forecaster_lasso.regressor.named_steps['lasso'].coef_
print("Lasso coefficients")
print(lasso_coefs)
ridge_coefs = forecaster_ridge.regressor.named_steps['ridge'].coef_
print("Ridge coefficients")
print(ridge_coefs)


print(f"Lasso: { (lasso_coefs == 0).sum() } coefficients set to zero out of { len(lasso_coefs) }")
print(f"Ridge: { (ridge_coefs == 0).sum() } coefficients set to zero out of { len(ridge_coefs) }")

##### **Exercise 4.3**

1.  Analyze the results of the code above:

    1.  Why might the Lasso model set several coefficients to exactly zero while the Ridge model keeps all of them?

    2.  If you were to use a very large $\lambda$, how would the forecast line behave visually?

2.  Modify the code in the example to include rolling, fourier and calendar features generated in the previous section.

#### 3.1.2. Selecting the Optimal $\lambda$ (Hyperparameter Tuning)

The regularization parameter $\lambda$ is a hyperparameter, meaning it must be defined before the training process begins. Since $\lambda$ controls the trade-off between model complexity and fitting quality, its selection is critical:

-   If $\lambda$ is too small, the model approaches the OLS solution, risking high variance and overfitting.

-   If $\lambda$ is too large, the coefficients are shrunk excessively, leading to high bias and underfitting.

Following a machine learning approach, the optimal $\lambda$ can be obtained via Backtesting with a grid of potential values (usually on a logarithmic scale: $\ldots, 10^{-2}, 10^{-1}, 10^{0}, 10^{1}, 10^{2}, \ldots = \ldots, 0.01, 0.1, 1, 10, 100, \ldots$).

To ensure a fair evaluation and avoid optimization bias, the data must be handled through a hierarchical scheme:

1.  Grid Search (Internal Backtesting): During this phase, each training fold is internally split into two parts:

    -   Sub-Train: Used to estimate the coefficients ($\beta$) for a specific $\lambda$ value.

    -   Validation: Used to evaluate the error (e.g., RMSE) produced by that specific $\lambda$.

        The $\lambda$ that yields the lowest average error across all validation folds is selected as the optimal hyperparameter.

2.  Model Refit: Once the optimal $\lambda$ is identified, the model is retrained using the entire training dataset to obtain the final, most robust set of coefficients ($\beta$).

3.  Final Evaluation: Finally, the predictive performance of this optimized model is measured on a Test Set (a block of data that was never used during the Grid Search). This provides an honest assessment of the model's quality using the accuracy metrics introduced in Chapter 2.

The following code implements the full Train-Validation-Test workflow searching for the best $\lambda$ (`alpha` parameter in the `skforecast` implementation) using backtesting on the training data and then evaluates the final "winner" on the held-out test set.

In [ ]:
#| label: lambda-tuning-skforecast
#| echo: true

import numpy as np
from skforecast.model_selection import grid_search_forecaster, TimeSeriesFold

# 1. Outer Split: Separate the 'Final Exam' (Test Set)
# We use the rest for Tuning (Train + Validation)
steps_test = 24
y_train = data['sales'][:-steps_test]
y_test  = data['sales'][-steps_test:]

# 2. Define the Pipeline and Forecaster
pipe = make_pipeline(StandardScaler(), Ridge())
forecaster = ForecasterRecursive(regressor=pipe, lags=24)

# 3. Define the Search Grid (Logarithmic Scale)
# This creates 10 values from 10^-3 (0.001) to 10^3 (1000)
param_grid = {'ridge__alpha': np.logspace(-3, 3, 10)}

# 4. Define Cross-Validation Strategy with TimeSeriesFold
cv = TimeSeriesFold(
    steps              = 24, # Forecasting horizon for validation
    initial_train_size = len(y_train) - 36, # Sub-train size
    fold_stride        = 1, # steps the sliding window moves forward (by default = steps)
    refit              = True, # Refit model at each fold (=False: no refit; =3: more efficient refit each 3 steps)
    fixed_train_size   = False,              # Expanding window
    gap                = 0,                  # No gap between train and test
    allow_incomplete_fold = True
)

# 5. Run Grid Search with Backtesting
results_grid = grid_search_forecaster(
    forecaster    = forecaster,
    y             = y_train,
    param_grid    = param_grid,
    cv            = cv,                      
    metric        = 'mean_absolute_error',
    return_best   = True, # Keeps the best model in the forecaster object
    n_jobs        = 'auto', #parallel evaluation if more cores ar available
    verbose       = False,
    show_progress = True
)

print("Grid Search Results (Top 3):")
print(results_grid.head(3))
print(f"Best lambda value: {results_grid.iloc[0].ridge__alpha}")

# 5. Final Prediction on Test Set
# Since refit=True, 'forecaster' now holds the best alpha and is trained on y_train
predictions = forecaster.predict(steps=24)

# 6. Visualization
fig, ax = plt.subplots(figsize=(12, 6))
data_tail = y_train.tail(50)
ax.plot(data_tail.index.to_pydatetime(), data_tail.values, label='Training Data (Tail)', color='gray', alpha=0.5)
predictions.plot(ax=ax, label='Ridge Forecast (L2)', color='blue', linestyle='--')

ax.set_title("Sales Forecast (Ridge Regularization)")
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()

##### **Exercise 4.4:**

1.  Modify the code in the example to select the optimal hypermaramter for a Lasso model. Be careful with the parameter name, it change with the model name.

2.  Modify the code in the example to include rolling, fourier and calendar features generated in the previous section. Consult python documentation (`help(grid_search_forecaster)`) to know how to deal with exogenous variables.

## 3.2. Tree-Based Models

Tree-based ensembles, such as Random Forest and Gradient Boosting, have become a cornerstone of modern machine learning due to their exceptional predictive power and flexibility. While linear models (Ridge/Lasso) attempt to fit a global "best-fit" equation to the data, tree-based algorithms take a non-parametric approach. Instead of assuming a specific functional form, tree-based models use recursive partitioning to divide the feature space into hierarchical regions based on "if-then" rules.

### 3.2.1. Key Characteristics and Advantages

Tree-based models are often the first choice for complex forecasting tasks because they simplify the data preparation process while handling intricate patterns. Their primary advantages include:

-   Non-Linearity and Interaction Detection: Unlike linear regression, which requires manual creation of interaction terms (e.g., $X_1 \cdot X_2$) or polynomial features to handle non-linear curves, trees capture these relationships naturally. They can easily detect that the effect of a specific lag might change depending on the month of the year or a rolling average threshold.

-   Scale Invariance: This is a major practical advantage over Ridge and Lasso. Since trees split data based on ranked thresholds (e.g., "Is Lag 1 \> 150.5?**"**), the absolute magnitude of the variables is irrelevant. Consequently, feature scaling (standardization) is not required, making the models less sensitive to the units of measurement.

-   Robustness to Multicollinearity and Outliers:

    -   Multicollinearity: In our context of highly correlated time-lags, a tree simply selects the most informative predictor at each node split, whereas OLS-based models would suffer from unstable coefficients.

    -   Outliers: Extreme values only affect the specific leaf nodes where they fall, preventing a single outlier from "tilting" the entire model's logic as it would in a linear regression.

### 3.2.2. Main Limitations

While tree-based models offer great flexibility, their mathematical structure imposes significant constraints when forecasting continuous time series, particularly those with trends or high precision requirements.

#### A. The Extrapolation Barrier

The most critical limitation of tree models is their inability to extrapolate beyond the range of the training data. A tree predicts the value of a new observation by calculating the average of the target values in the leaf node where that observation falls. As a consequence, the model's output is strictly bounded by the minimum and maximum values seen during training ($y_{min} \leq \hat{y} \leq y_{max}$). If a time series has a strong upward or downward trend, the model output saturates once it reaches the edge of its historical data, as it cannot conceive of values it has never seen.

#### B. Discontinuity

Tree-based algorithms produce a discontinuous step function rather than a smooth, continuous trajectory. Because the feature space is partitioned into discrete hierarchical regions, the forecast remains constant within a specific "box" and shifts abruptly only when a predictor crosses a certain threshold. While ensembling methods like Random Forest can soften these transitions by averaging hundreds of trees, the underlying prediction remains piecewise constant, which may contrast with the fluid evolution typical of many economic or physical processes.

#### C. Sensitivity to Data Density

Tree-base models also exhibit a high sensitivity to data density in the feature space. In regions where training observations are sparse—such as rare combinations of lags or extreme historical events—the resulting averages in the leaf nodes can be easily biased by a single outlier. This makes the model's performance in high-dimensional spaces (like those involving numerous exogenous variables) dependent on having a sufficiently large and representative dataset to ensure stable local averages.

### 3.2.3. Random Forest

The Random Forest is an ensemble learning method that operates by constructing a multitude of decision trees during training. It belongs to the Bagging (Bootstrap Aggregating) family, and its main objective is to reduce the variance of individual decision trees—which are prone to overfitting—without increasing their bias.

![Random Forest](Figures/Chapter4/randomForest.png){fig-align="center"}

The power of Random Forest lies in two main concepts:

1.  **Bagging:** Each tree in the forest is trained on a random subsample of the data (with replacement). This ensures that each tree "sees" a slightly different version of the historical series.

2.  **Feature Randomness:** When splitting a node, the algorithm usually does not evaluate all available predictors. Instead, it selects a random subset of features to find the best split. This decorrelates the trees; if one lag is extremely dominant, not all trees will rely on it, allowing the forest to discover more subtle patterns. A widely used rule of thumb in machine learning is to consider only the square root of the total number of features ($k \approx \sqrt{p}$) at each node. However, hyperparemeter tuning guided by backtesting is also usual.

The final forecast is the average of the predictions from all individual trees. Mathematically, for $B$ trees, the ensemble prediction $\hat{y}$ is:

$$
\hat{y} = \frac{1}{B} \sum_{b=1}^{B} f_b(x)
$$

#### A. Implementation in Python

The behavior of a Random Forest is controlled by several structural hyperparameters. Choosing the right values is critical to balance the model's ability to learn patterns against its tendency to memorize noise.

1.  **`n_estimators`** (Number of Trees): Controls the stability of the model. Generally, more trees lead to a more robust average and lower variance. Unlike other parameters, increasing this rarely causes overfitting, but improvements follow a law of diminishing returns. *Typical values* start with 100 but values between 200 and 500 are common.

2.  **`max_depth`** (Tree Complexity): Limits the number of splits in each tree. This hyperparameter is critical for preventing overfitting. A tree that is too deep will memorize the training noise but, by contrast, a tree with too few levels is not able to learn the patterns needed to predict accurately. A tree depth between 5 and 20 is usual. Although for small datasets, lower values (3-10) are preferred.

3.  **`max_features`** (Decorrelation): The number of features considered at each split. By restricting the maximum number of features we force tree diversity. *Typical values are* `'sqrt'` (square root of total features), `'log2'` (base 2 logarimth of total features), or a float (e.g., `0.3` representing that 30% of total features are considered) are also common alternatives.

Additionally, as the Random Forest depends on random sampling, `random_state` allows to control the random seed to allow reproducibility.

**Basic Implementation** Since Random Forest is scale-invariant, we can pass the regressor directly to the forecaster without a `StandardScaler`.

In [ ]:
#| label: random-forest-implementation
#| echo: true

from sklearn.ensemble import RandomForestRegressor
from skforecast.recursive import ForecasterRecursive

# 1. Initialize the Regressor
# We set n_estimators=100 and max_depth=10 as a solid baseline
rf_regressor = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)

# 2. Create and Fit Forecaster
forecaster_rf = ForecasterRecursive(regressor=rf_regressor, lags=24)
forecaster_rf.fit(y=y_train)

# 3. Predict
pred_rf = forecaster_rf.predict(steps=24)

In [ ]:
#| label: fig-rf-forecast
#| fig-cap: Random Forest Forecast (24 months) vs. Actual Data. Note the limitations in trend extrapolation compared to the actual test values.
#| echo: false

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))

# Graficamos el final del entrenamiento para dar contexto
y_train.tail(50).plot(ax=ax, label='Training Data (Tail)', color='gray', alpha=0.5)

# Graficamos los datos reales del test
y_test.plot(ax=ax, label='Actual Values (Test)', color='black', linewidth=1.5)

# Graficamos la predicción
pred_rf.plot(ax=ax, label='Random Forest Forecast', color='green', linestyle='--')

ax.set_title("Random Forest Forecast vs. Actual Data")
ax.set_ylabel("Sales")
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Optimization via Grid Search** When tuning a small number of critical parameters (e.g., `max_depth` and `max_features`), an exhaustive grid search is the most reliable strategy. It evaluates every possible combination in the provided dictionary.

In [ ]:
#| label: rf-grid-search
#| echo: true

from sklearn.ensemble import RandomForestRegressor
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import grid_search_forecaster, TimeSeriesFold

# 1. Create the Forecaster
forecaster_rf = ForecasterRecursive(
    regressor = RandomForestRegressor(random_state=42),
    lags = 24  # Placeholder, will be overwritten if using lags_grid
)

# 2. Define the Grid
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, 15],
    'max_features': ['sqrt', 1.0] 
}

# 3. Configure Time Series Cross-Validation with TimeSeriesFold
initial_train_size = len(y_train) - 36
steps = 24 # Forecasting horizon for validation
cv = TimeSeriesFold(
    steps              = steps,
    initial_train_size = initial_train_size, # Sub-train size
    fold_stride        = 1, # steps the sliding window moves forward (by default = steps)
    refit              = 5, # Refit model at each 5 folds
    fixed_train_size   = False,              # Expanding window
    gap                = 0,                  # No gap between train and test
    allow_incomplete_fold = True
)

# 4. Run Grid Search
results_grid_rf = grid_search_forecaster(
    forecaster   = forecaster_rf,
    y            = y_train,
    param_grid   = param_grid,
    cv           = cv,
    metric       = 'mean_absolute_error',
    return_best  = True,
    n_jobs       = 'auto',
    verbose      = False,
    show_progress = True
)

cv.verbose = False 
n_folds = len(list(cv.split(y_train)))
print(f"Grid Search performed over {n_folds} folds of {steps} months.")
print("Best Parameters (Grid Search):")
print(results_grid_rf.iloc[0]['params'])
best_mae = results_grid_rf.iloc[0]['mean_absolute_error']
print(f"Best MAE: {best_mae}")

# 5. Predict
pred_rf = forecaster_rf.predict(steps=24)

In [ ]:
#| label: fig-rf-grid_search_tunning-forecast
#| fig-cap: Random Forest with hyperparameter tunning (grid-search) vs. Actual Data.
#| echo: false

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))

# Graficamos el final del entrenamiento para dar contexto
y_train.tail(50).plot(ax=ax, label='Training Data (Tail)', color='gray', alpha=0.5)

# Graficamos los datos reales del test
y_test.plot(ax=ax, label='Actual Values (Test)', color='black', linewidth=1.5)

# Graficamos la predicción
pred_rf.plot(ax=ax, label='Random Forest Forecast', color='green', linestyle='--')

ax.set_title("Random Forest Forecast vs. Actual Data")
ax.set_ylabel("Sales")
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Optimization via Random Search** When the hyperparameter space is large (e.g., tuning lag size, depth, features, and split criteria simultaneously), the number of combinations grows exponentially, making Grid Search computationally prohibitive.

In these cases, Random Search is preferred. Instead of testing all combinations, it samples a fixed number of configurations (`n_iter`) from the defined distributions. This allows for a much broader exploration of the search space with a controlled computational budget.

In [ ]:
#| label: rf-random-search
#| echo: true
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import random_search_forecaster, TimeSeriesFold

# 1. Create Forecaster
forecaster_rf = ForecasterRecursive(
    regressor = RandomForestRegressor(random_state=42),
    lags = 24
)

# 2. Define the Parameter Distributions
# In Random Search, we can provide larger lists or distributions
param_distributions = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [5, 10, 15, 20, 25, None],
    'max_features': ['sqrt', 'log2', None],
    'min_samples_split': [2, 5, 10], 
    'min_samples_leaf': [1, 2, 4]    
}

# 3. Configure Time Series Cross-Validation with TimeSeriesFold
initial_train_size = len(y_train) - 36


# 3. Define Cross-Validation Strategy with TimeSeriesFold
steps = 24 # Forecasting horizon for validation
cv = TimeSeriesFold(
    steps              = steps, 
    initial_train_size = initial_train_size, # Sub-train size
    fold_stride        = 1, # steps the sliding window moves forward (by default = steps)
    refit              = 5, # Refit model each 5 folds 
    fixed_train_size   = False,              # Expanding window
    gap                = 0,                  # No gap between train and test
    allow_incomplete_fold = True
)

# 4. Run Random Search with Backtesting
n_iter=20 # randomly select and test only 20 combinations
results_random_rf = random_search_forecaster(
    forecaster          = forecaster_rf,
    y                   = y_train,
    param_distributions = param_distributions,
    n_iter              = n_iter,  # Computational budget (number of trials)
    cv                  = cv,
    metric              = 'mean_absolute_error',
    return_best         = True,
    random_state        = 42,
    n_jobs              = 'auto',
    verbose             = False,
    show_progress       = True
)

cv.verbose =False
n_folds = len(list(cv.split(y_train)))
print(f"Random Search performed over {n_folds} folds of {steps} steps.")
n_combinations = np.prod([len(v) for v in param_distributions.values()])
print(f"Searching {n_iter} random combinations from a total search space of {n_combinations}")
print("Best Parameters (Random Search):")
print(results_random_rf.iloc[0]['params'])
best_mae = results_random_rf.iloc[0]['mean_absolute_error']
print(f"Best MAE: {best_mae}")

# 5. Predict
pred_rf = forecaster_rf.predict(steps=24)

In [ ]:
#| label: fig-rf-random_search_tunning-forecast
#| fig-cap: Random Forest with hyperparameter tunning (random-search) vs. Actual Data.
#| echo: false

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))

# Graficamos el final del entrenamiento para dar contexto
y_train.tail(50).plot(ax=ax, label='Training Data (Tail)', color='gray', alpha=0.5)

# Graficamos los datos reales del test
y_test.plot(ax=ax, label='Actual Values (Test)', color='black', linewidth=1.5)

# Graficamos la predicción
pred_rf.plot(ax=ax, label='Random Forest Forecast', color='green', linestyle='--')

ax.set_title("Random Forest Forecast vs. Actual Data")
ax.set_ylabel("Sales")
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()

#### **Exercise 4.5: Forecasting Beer Production**

To consolidate the concepts of Bagging and Grid Search, try applying a Random Forest model to a new dataset.

-   Dataset: Australian Beer Production (`beerProduction_aus.xlsx`). This dataset records quarterly beer production.

-   Tune a Random Forest to predict the next 2 years (8 quarters). Pay attention to the frequency of the data (Quarterly).

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import grid_search_forecaster
import matplotlib.pyplot as plt

# 1. Load Data
df_beer = pd.read_excel('beerProduction_aus.xlsx')
df_beer['Date'] = pd.to_datetime(df_beer['Date'])
df_beer = df_beer.set_index('Date')
df_beer = df_beer.asfreq('QS-JAN')

# Train-Test Split (Last 8 quarters for testing)
train_size = len(df_beer) - 8
y_train = df_beer.iloc[:train_size, 0]
y_test = df_beer.iloc[train_size:, 0]

# 2. Define the Forecaster
forecaster_rf_beer = ForecasterRecursive(
    regressor = RandomForestRegressor(random_state=42),
    lags      = 4
)

# 3. Parameters for Grid Search
param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [3, 5, 10, None]
}
lags_grid = [4, 8, 12] 

# 4. Grid Search
results_grid_beer = grid_search_forecaster(
    forecaster         = forecaster_rf_beer,
    y                  = y_train,
    param_grid         = param_grid,
    lags_grid          = lags_grid,
    steps              = 8,
    metric             = 'mean_absolute_error',
    refit              = False,
    initial_train_size = len(y_train) - 16,
    return_best        = True,
    verbose            = False,
    show_progress      = False
)

print(f"Best Parameters: {results_grid_beer.iloc[0]['params']}")
print(f"Best Lags: {results_grid_beer.iloc[0]['lags']}")

# 5. Predict
pred_beer = forecaster_rf_beer.predict(steps=8)

# 6. Plot
fig, ax = plt.subplots(figsize=(10, 5))
y_train.tail(40).plot(ax=ax, label='Train', color='gray', alpha=0.5)
y_test.plot(ax=ax, label='Test (Actual)', color='black')
pred_beer.plot(ax=ax, label='Random Forest Forecast', color='green', linestyle='--')
ax.set_title("Quarterly Beer Production Forecast (Random Forest)")
ax.set_ylabel("Beer Production")
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 3.2.4. XGBoost (Extreme Gradient Boosting)

While Random Forest relies on averaging independent trees to reduce variance (Bagging), XGBoost (Extreme Gradient Boosting) follows a completely different philosophy called Boosting.

In this approach, trees are built sequentially, not in parallel. Each new tree is trained to correct the errors (residuals) made by the previous ensemble of trees. Instead of a "democracy" of independent trees, XGBoost acts like a team where each new member focuses solely on fixing the mistakes of the team so far.

The power of XGBoost lies in two main concepts:

1\. Sequential Correction (Learning from Residuals)

-   Iteration 1: The model trains the first tree ($f_1$) on the original data $(x, y)$.

-   Iteration 2: The model calculates the residuals ($r_1 = y - \hat{y}^{(1)}$) and trains a second tree ($f_2$) not to predict $y$, but to predict these residuals.

-   Iteration b: This process repeats up to $B$ trees. Each new tree attempts to predict the error remaining from the combination of all previous trees.

The model is additive. The forecast is given by the prediction of the first tree plus the weighted sum of the corrections predicted by the rest of the trees:

$$
\hat{y}_i = f_1(x_i)+ \eta \sum_{b=2}^{B} f_b(x_i)
$$

Similarly, we can write the prediction given by the $b$-th tree as a recursive formula:

$$
\hat{y}_i^{(b)} = \hat{y}_i^{(b-1)} + \eta f_b(x_i)
$$

Where $\eta$ is the learning rate, which scales the contribution of each new tree to prevent overfitting.

![XGBoost algorithm](Figures/Chapter4/GXBoost.png)

To build the optimal tree at each step, XGBoost optimizes a specific objective function that balances accuracy (Loss) and simplicity (Regularization). For the $b$-th tree, the objective function $\mathcal{L}^{(b)}$ to minimize is:

$$
\mathcal{L}^{(b)} = \underbrace{\sum_{i=1}^{n} l(y_i, \hat{y}_i^{(b)})}_{\text{Loss Function}} + \underbrace{\Omega(f_b)}_{\text{Regularization}}
$$

-   Loss Function ($l$): Measures how well the model predicts the training data (e.g., Mean Squared Error). XGBoost uses a second-order Taylor expansion (using both gradients and Hessians) to approximate this loss, allowing for faster and more accurate convergence than standard Gradient Boosting.

-   Regularization ($\Omega$): This term penalizes complex trees structures to prevent overfitting and also introduces its L1 and L2 regularization by penalizing magnitude of its leaf weights ($w$):

    $$
    \Omega(f_b) =  \underbrace{\gamma T}_{\text{Tree Structure}} + \underbrace{\frac{1}{2} \lambda ||w||^2}_{\text{L2 Reg.}} + \underbrace{\alpha ||w||_1}_{\text{L1 Reg.}}
    $$

    where $T$ is the number of leaves, and the vector of scores assigned to the leaves is given by $w \in \mathbb{R}^T$. For a specific instance $x_i$, the tree output is the weight of the assigned leaf: $f_b(x_i) = w_{q(x_i)}$.

    Crucially, regularization is integrated directly into the calculation of the optimal weights. By minimizing the objective function, the optimal weight $w_j^*$ for a specific leaf $j$ is derived from the gradients ($G$) and Hessians ($H$) of the instances in that leaf:

    $$
    w_j^* = - \frac{\text{SoftThreshold}(\sum G, \alpha)}{\sum H + \lambda}
    $$

    Where the Soft Thresholding function is defined as:

    $$
    \text{SoftThreshold}(x, \alpha) =
    \begin{cases}
    x - \alpha & \text{if } x > \alpha \\
    0 & \text{if } |x| \le \alpha \\
    x + \alpha & \text{if } x < -\alpha
    \end{cases}
    $$

    This formula reveals the direct mechanical impact of the hyperparameters:

    -   L2 Regularization (Ridge, $\lambda$): Adds to the denominator. As $\lambda$ increases, the denominator grows, "shrinking" the weight $w_j^*$ towards zero. This prevents any single leaf from having an aggressively high score when the evidence (Hessian) is low. It is the default setting ($\lambda=1$).

    -   L1 Regularization (Lasso, $\alpha$): Acts as a threshold on the numerator. If the accumulated gradient ($\sum G$) is smaller than $\alpha$, the weight is forced to exactly zero. This induces sparsity, effectively performing feature selection by pruning unnecessary parts of the tree.

    -   $\gamma$ (Gamma): Acts as a minimum loss reduction threshold. If the gain from splitting a node does not exceed $\gamma$, the split is discarded (pruning).

    Since the optimal magnitude of $\lambda$ and $\alpha$ depends on the scale of the target variable and the gradients, fixing a narrow search range a priori is risky. Therefore, we employ a Random Search over a broad, logarithmically spaced grid (e.g., $10^{-2}, \dots, 10^{2}$) to ensure the search space encompasses the appropriate scale for the dataset.

#### **A. Implementation in Python**

Because XGBoost builds trees sequentially, its hyperparameters interact differently than in Random Forest. Balancing the "learning speed" against complexity is key.

-   **`learning_rate` (**$\eta$**):** Controls the contribution of each tree. A lower rate (e.g., 0.01) makes the model more robust to overfitting but requires more trees (`n_estimators`) to learn the pattern.

-   **`n_estimators`:** The number of sequential trees. Unlike Random Forest, increasing this too much can lead to overfitting if the learning rate is high, as the model starts memorizing the noise in the residuals.

-   **`max_depth`:** Depth of the individual trees. In Boosting, trees are typically "weak learners" (shallow trees: typical values are 3 to 6).

-   **`reg_alpha` (L1) & `reg_lambda` (L2):** Control the regularization discussed above.

**Basic Implementation**

We utilize the `XGBRegressor` class, which is fully compatible with `skforecast`.

In [ ]:
#| label: xgboost-implementation
#| echo: true

from xgboost import XGBRegressor
from skforecast.recursive import ForecasterRecursive

# 1. Initialize the Regressor
# We use a conservative learning rate (0.1) and shallow trees (depth=6)
xgb_regressor = XGBRegressor(
    n_estimators=100, 
    max_depth=6, 
    learning_rate=0.1, 
    # Regularization parameters (matching the theoretical definitions)
    reg_alpha=0,      # L1 Regularization (alpha) - Default: 0
    reg_lambda=1,     # L2 Regularization (lambda) - Default: 1
    gamma=0,          # Min loss reduction (gamma) - Default: 0
    random_state=42
)

# 2. Create and Fit Forecaster
forecaster_xgb = ForecasterRecursive(regressor=xgb_regressor, lags=24)
forecaster_xgb.fit(y=y_train)

# 3. Predict
pred_xgb = forecaster_xgb.predict(steps=24)

In [ ]:
#| label: fig-xgb-forecast
#| fig-cap: XGBoost Forecast (Basic) vs. Actual Data.
#| echo: false

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))

# Plotting the last 50 observations of the training data
y_train.tail(50).plot(ax=ax, label='Training Data (Tail)', color='gray', alpha=0.5)

# Plotting the actual test data
y_test.plot(ax=ax, label='Actual Values (Test)', color='black', linewidth=1.5)

# Plotting the XGBoost predictions
pred_xgb.plot(ax=ax, label='XGBoost Forecast', color='blue', linestyle='--')

ax.set_title("XGBoost Forecast vs. Actual Data")
ax.set_ylabel("Sales")
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Optimization via Grid Search**

For XGBoost, the relationship between `learning_rate` and `n_estimators` is the most critical interaction to tune. We use `TimeSeriesFold` to validate performance over 3 different periods.

In [ ]:
#| label: xgb-grid-search
#| echo: true

from skforecast.model_selection import grid_search_forecaster, TimeSeriesFold

# 1. Create the Forecaster
forecaster_xgb = ForecasterRecursive(
    regressor = XGBRegressor(random_state=42),
    lags = 24  
)

# 2. Define the Grid
# We test a "slow and steady" approach vs a "fast" approach
param_grid = {
    'n_estimators': [100, 500],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 6] # Shallow vs Medium depth
}

# 3. Configure Time Series Cross-Validation with TimeSeriesFold
initial_train_size = len(y_train) - 36
steps = 24 

cv = TimeSeriesFold(
    steps              = steps,
    initial_train_size = initial_train_size, 
    fold_stride        = 1, 
    refit              = 5, 
    fixed_train_size   = False,              
    gap                = 0,                  
    allow_incomplete_fold = True
)

# 4. Run Grid Search
results_grid_xgb = grid_search_forecaster(
    forecaster   = forecaster_xgb,
    y            = y_train,
    param_grid   = param_grid,
    cv           = cv,
    metric       = 'mean_absolute_error',
    return_best  = True,
    n_jobs       = 'auto',
    verbose      = False,
    show_progress = True
)

cv.verbose = False
n_folds = len(list(cv.split(y_train)))
print(f"Grid Search performed over {n_folds} folds of {steps} months.")
print("Best Parameters (Grid Search):")
print(results_grid_xgb.iloc[0]['params'])
best_mae = results_grid_xgb.iloc[0]['mean_absolute_error']
print(f"Best MAE: {best_mae}")

# 5. Predict
pred_xgb = forecaster_xgb.predict(steps=24)

**Optimization via Random Search**

XGBoost involves a large number of hyperparameters. Beyond the regularization terms ($\gamma$, $\alpha, \lambda$) discussed previously, we introduce two additional parameters to the search space that control the model's complexity and its stochastic nature:

1.  Subsample (`subsample`): Controls the fraction of training instances (rows) sampled randomly to train each individual tree. Setting this to a value less than 1.0 (e.g., 0.7) means that XGBoost randomly collects 70% of the data for each tree. This technique, known as Stochastic Gradient Boosting, prevents the model from overfitting to specific data points.

2.  Colsample by Tree (`colsample_bytree`): Controls the fraction of features (columns) sampled randomly for each tree. Similar to Random Forests, this helps to decorrelate the trees and makes the model robust to noise in specific features.

Since the optimal scale for regularization is unknown, we employ a log-scale search strategy for `reg_alpha`, `reg_lambda`, and `gamma` to cover different orders of magnitude.

In [ ]:
#| label: xgb-random-search
#| echo: true

import numpy as np
from skforecast.model_selection import random_search_forecaster, TimeSeriesFold
from xgboost import XGBRegressor
from skforecast.recursive import ForecasterRecursive

# 1. Create Forecaster
forecaster_xgb = ForecasterRecursive(
    regressor = XGBRegressor(random_state=42),
    lags = 24
)

# 2. Define the Parameter Distributions
# We include Gamma and Stochastic parameters (subsample, colsample).
# Regularization terms use a geometric progression to cover multiple scales.
param_distributions = {
    'n_estimators': [100, 300, 500, 1000],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6, 8, 10],
    
    # Stochastic Gradient Boosting (Randomness to reduce variance)
    'subsample': [0.5, 0.7, 1.0],        # % of rows used per tree
    'colsample_bytree': [0.5, 0.7, 1.0], # % of features used per tree
    
    # Structural Pruning (Minimum Loss Reduction)
    'gamma': [0, 0.1, 1.0, 5.0, 10.0],   
    
    # L1 & L2 Regularization (Log-scale search)
    'reg_alpha': [0, 0.001, 0.01, 0.1, 1, 10],      # L1 (Lasso)
    'reg_lambda': [0.001, 0.01, 0.1, 1, 10]         # L2 (Ridge)
}

# 3. Configure Time Series Cross-Validation
initial_train_size = len(y_train) - 36
steps = 24 

cv = TimeSeriesFold(
    steps              = steps, 
    initial_train_size = initial_train_size, 
    fold_stride        = 1, 
    refit              = 5, 
    fixed_train_size   = False,              
    gap                = 0,                  
    allow_incomplete_fold = True
)

# 4. Run Random Search
n_iter = 50 # Increased to 50 to cover the larger search space
results_random_xgb = random_search_forecaster(
    forecaster          = forecaster_xgb,
    y                   = y_train,
    param_distributions = param_distributions,
    n_iter              = n_iter,
    cv                  = cv,
    metric              = 'mean_absolute_error',
    return_best         = True,
    random_state        = 42,
    n_jobs              = 'auto',
    verbose             = False,
    show_progress       = True
)

cv.verbose = False
n_folds = len(list(cv.split(y_train)))
print(f"Random Search performed over {n_folds} folds of {steps} steps.")

n_combinations = np.prod([len(v) for v in param_distributions.values()])
print(f"Searching {n_iter} random combinations from a total search space of {n_combinations}")

print("Best Parameters (Random Search):")
print(results_random_xgb.iloc[0]['params'])
best_mae = results_random_xgb.iloc[0]['mean_absolute_error']
print(f"Best MAE: {best_mae}")

# 5. Predict
pred_xgb = forecaster_xgb.predict(steps=24)

In [ ]:
#| label: fig-xgb-random-search-forecast
#| fig-cap: XGBoost with hyperparameter tuning (Random Search) vs. Actual Data.
#| echo: false

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))
y_train.tail(50).plot(ax=ax, label='Training Data (Tail)', color='gray', alpha=0.5)
y_test.plot(ax=ax, label='Actual Values (Test)', color='black', linewidth=1.5)
pred_xgb.plot(ax=ax, label='XGBoost Forecast', color='blue', linestyle='--')
ax.set_title("XGBoost Forecast vs. Actual Data")
ax.set_ylabel("Sales")
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()

#### Exercise 4.6: US Retail Employment

Apply the advanced regularization and log-scale tuning strategies of XGBoost to a monthly economic dataset.

-   Dataset**:** US Retail Employment (`us_retail_employment.xlsx`).

-   Optimize an XGBoost model to forecast the next year (12 months) of employment levels. Use Random Search to explore orders of magnitude for `reg_alpha` and `reg_lambda`. Include `subsample` to test stochastic gradient boosting.

### 3.2.5 Simple Neural Networks (MLPs)

While sequence-specific architectures like LSTMs or Transformers are the gold standard for deep learning in time series, traditional Multi-Layer Perceptrons (MLPs) serve as a powerful bridge between classical Machine Learning and Deep Learning.

In the context of `skforecast`, an MLP treats the time series problem as a standard regression task using a tabular format. However, unlike tree-based models, Neural Networks interpret inputs geometrically. This requires careful feature engineering:

-   Lags: Past values of the series ($y_{t-1}, y_{t-2}, \dots$).

-   Window Features: Aggregated statistics (rolling mean, std) to capture trends and volatility.

-   Cyclical Calendar Features: Unlike Decision Trees, MLPs struggle with raw calendar numbers (e.g., Months 1 to 12). For an MLP, the mathematical distance between December (12) and January (1) is large (11), creating an artificial discontinuity. To reflect the cyclical nature of time, calendar features should be transformed into coordinates on a circle using sine and cosine functions as seen in Section 2.3.4.:

    $$
    x_{sin} = \sin\left(\frac{2\pi \cdot t}{T}\right), \quad x_{cos} = \cos\left(\frac{2\pi \cdot t}{T}\right)
    $$

    Where $T$ is the period (e.g., 12 for months, 7 for days of the week, ...).

$$
x = [\underbrace{y_{t-1}, \dots, y_{t-p}}_{\text{Lags}}, \underbrace{\mu_{t-7}}_{\text{Roll}}, \underbrace{\sin(\text{Mo}_t), \cos(\text{Mo}_t)}_{\text{Cyclical Exog}}] \xrightarrow{\text{MLP}} \hat{y}_{t}
$$

Key Considerations for Time Series:

1.  Scaling is Mandatory: MLPs are highly sensitive to the scale of inputs. Large values cause gradients to explode or vanish. All inputs (lags and exogenous variables) must be scaled (e.g., `StandardScaler`) before entering the network.

2.  Stationarity: Standard MLPs do not extrapolate trends well. Strong trends should be removed (detrending) or the series should be differenced.

3.  Cyclical Encoding**:** As mentioned, raw ordinal time features (1-12) should be avoided in favor of sine/cosine transformations to preserve temporal proximity.

4.  Overfitting: Techniques like Early Stopping and L2 Regularization (Alpha) are essential.

#### A. Implementation with `skforecast`

To ensure that all features are properly scaled, we wrap the `MLPRegressor` inside a `sklearn.pipeline.make_pipeline`.

In [ ]:
#| label: mlp-implementation
#| echo: true

from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from skforecast.recursive import ForecasterRecursive

# 1. Define the Regressor as a Pipeline
# It is crucial to scale the data before feeding them to the MLP.
mlp_pipeline = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=(64, 32), 
        activation='relu',           
        solver='adam',               
        learning_rate_init=0.01,
        max_iter=500,
        early_stopping=True,         
        random_state=42
    )
)

# 2. Create and Fit Forecaster
forecaster_mlp = ForecasterRecursive(regressor=mlp_pipeline, lags=24)
forecaster_mlp.fit(y=y_train)

# 3. Predict
pred_mlp = forecaster_mlp.predict(steps=24)

In [ ]:
#| label: fig-mlp-forecast
#| fig-cap: MLP Forecast vs. Actual Data.
#| echo: false

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))
y_train.tail(50).plot(ax=ax, label='Training Data (Tail)', color='gray', alpha=0.5)
y_test.plot(ax=ax, label='Actual Values (Test)', color='black', linewidth=1.5)
pred_mlp.plot(ax=ax, label='MLP Forecast', color='blue', linestyle='--')
ax.set_title("MLP Forecast vs. Actual Data")
ax.set_ylabel("Sales")
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()

#### Exercise 4.7: Tuning the Neural Network

Neural Networks are notoriously sensitive to their hyperparameters. A configuration that works perfectly for one dataset may fail completely for another (e.g., the gradients might vanish, or the model might get stuck in a local minimum).

Use the `h2o` dataset and perform a Random Search to find the optimal architecture for predicting the next 24 months. The goal is to improve the prediction compared to the previous model where the hyperparemeters were arbitrary fixed.

Parameters to tune:

-   `hidden_layer_sizes`: Control the capacity of the network.

    -   *Search Space:* Try simple architectures `(50,)`, `(100,)` and deeper ones `(50, 25)`, `(100, 50)`.

-   `learning_rate_init`: The step size for the optimizer. This is often the most critical hyperparameter. Use a wide search space in logarithmic scale `[0.001, 0.01, 0.1]`.

-   `alpha` (L2 Regularization): Penalties on weights to prevent overfitting. Use a search space in logarithmic scale `[0.0001, 0.001, 0.01]`.

-   `activation`: The non-linear function. While `relu` activation function is the standard value, `tanh` also may work properly in time series data. `['relu', 'tanh']`.

> Important Hint (Pipelines): When tuning a model wrapped in a `Pipeline`, the parameter names in the grid must include the name of the step followed by a double underscore. Since we named our MLP step `'mlp'`, the parameter `hidden_layer_sizes` becomes `mlp__hidden_layer_sizes`.

## 4. Prophet: Additive Regression Models for Business Forecasting

Prophet is an open-source forecasting library developed by Meta (Facebook) aimed at producing high-quality forecasts for business time series that have strong seasonal effects and historical data spanning several seasons. Unlike the "black-box" nature of some machine learning models, Prophet is designed to be interpretable and tunable by analysts with domain expertise.

Conceptually, Prophet sits at the intersection between classical statistical models (like ARIMA or Exponential Smoothing) and pure Machine Learning algorithms (like XGBoost or Neural Networks):

-   Like Statistical Models (White-Box): It is fully explanatory. Its predictions are not generated by opaque weights, but by the sum of clear components. An analyst can visually inspect each component to understand *why* the model predicts a certain value.

-   Like Machine Learning Models: Under the hood, it abandons the rigid inferential assumptions of classical statistics. Instead, it employs modern data science methodologies such as L1/L2 regularization (to perform feature selection on changepoints and control seasonality complexity), generative sampling (for uncertainty estimation), and rigorous hyperparameter tuning via cross-validation.

In essence, Prophet offers the interpretability of statistics with the flexibility and fitting robustness of Machine Learning.

**Strategic Applicability: Strengths and Limitations**

Given its structural characteristics, Prophet is not a universal solution. It is specifically optimized for business and operational forecasting tasks where human interpretation is key.

Prophet excels in scenarios with:

-   Strong multiple seasonalities: For instance, hourly data that exhibits simultaneous daily, weekly, and yearly cycles.

-   Structural breaks and irregularities: Data with known outliers, trend shifts, or specific calendar events (holidays, promotions) that would violate the stationarity assumptions of ARIMA.

-   Missing Data: Unlike autoregressive models that require continuous data, Prophet handles gaps naturally as a curve-fitting problem.

However, Prophet is ill-suited for:

-   Purely autoregressive processes: If the value $y_t$ depends strongly and almost exclusively on the immediate past value $y_{t-1}$ (e.g., random walk stock prices), ARIMA or LSTM models will likely outperform Prophet, as Prophet focuses on the structural curve rather than immediate lag dependency.

-   Complex causal inference: Scenarios where exogenous variables interact non-linearly.

### 4.1. Mathematical Foundations

The Prophet model, introduced by Taylor and Letham (2017), falls within the class of Generalized Additive Models (GAM). Unlike ARIMA models, which rely on the autocorrelation structure of residuals (dependence of $y_t$ on $y_{t-k}$), Prophet approaches forecasting as a curve-fitting problem.

A major practical advantage of Prophet over classical statistical models is that Prophet does not require the time series to be stationary.

-   ARIMA requires stationarity (constant mean and variance) because it relies on the correlation between past and present values. Trend must be removed via differencing ($d$).

-   Prophet acts as a curve-fitting regression where time $t$ is the main regressor. It explicitly models non-stationarity through its trend component (handling changes in the mean) and multiplicative seasonality (handling changes in variance). Therefore, raw data with strong trends and seasonality can be fed directly into the model.

The analytical formulation decomposes the time series $y(t)$ into three main components of variation plus an error term:

$$
y(t) = g(t) + s(t) + h(t) + \epsilon_t
$$

Where:

-   $g(t)$: Represents the trend (non-periodic long-term growth or decline).

-   $s(t)$: Models seasonality (periodic patterns such as yearly, weekly, or daily cycles).

-   $h(t)$: Captures the effects of holidays or scheduled irregular events.

-   $\epsilon_t$: Is the error term, assumed to be normally distributed $\epsilon_t \sim \mathcal{N}(0, \sigma^2)$, representing idiosyncratic variations not explained by the model.

In the following sections we describe each component more specifically.

#### 4.1.1. The Trend Model ($g(t)$): Piecewise Linear Regression

Prophet models the trend $g(t)$ by assuming that the growth rate is not constant but changes at some specific points in time called changepoints. The default model is a Piecewise Linear Regression. Thus, the trend is defined as:

$$
g(t) = \underbrace{(k + \mathbf{a}(t)^\top \boldsymbol{\delta})}_{\text{Time-varying growth rate}} t + \underbrace{(m + \mathbf{a}(t)^\top \boldsymbol{\gamma})}_{\text{Time-varying offset}}
$$

Where:

-   $k$: The initial base growth rate.

-   $\boldsymbol{\delta} \in \mathbb{R}^S$: A vector containing the growth rate adjustments at the $S$ changepoints.

-   $\mathbf{a}(t) \in \{0, 1\}^S$: An indicator vector such that $a_j(t) = 1$ if $t \ge s_j$ (the time of changepoint $j$), and 0 otherwise.

-   $m$: The offset parameter (intercept).

-   $\boldsymbol{\gamma}$: A vector ensuring continuity of the function at the changepoints (preventing jumps in the trend line).

**Changepoint Selection and Regularization:**

A critical aspect of the model is the selection of changepoints $S$. Prophet specifies a large number of potential changepoints (default is 25) which are equidistantly distributed along the first 80% of the data. To select the significant changes among these candidates, the model employs a sparse Laplace prior for the adjustment coefficients $\delta_j$:

$$
\delta_j \sim \text{Laplace}(0, \tau)
$$

This Bayesian approach is equivalent to applying L1 Regularization (Lasso). The parameter $\tau$ (controlled by the hyperparameter `changepoint_prior_scale`) determines the model's sparsity:

-   A $\tau$ close to 0 forces most $\delta_j$ to be zero, resulting in a rigid trend (underfitting).

-   A large $\tau$ allows $\delta_j$ to take large values, resulting in a highly flexible trend (potential overfitting).

#### 4.1.2. Seasonality ($s(t)$): Approximation via Fourier Series

To model multi-period seasonality flexibly (e.g., simultaneous weekly and yearly patterns), Prophet uses an approximation based on Truncated Fourier Series.

For a seasonality of period $P$ (e.g., $P=365.25$ for yearly data), the component is modeled as:

$$
s(t) = \sum_{n=1}^{N} \left( a_n \cos\left(\frac{2\pi n t}{P}\right) + b_n \sin\left(\frac{2\pi n t}{P}\right) \right)
$$

-   Feature Engineering: Prophet transforms the time variable $t$ into a matrix of sine and cosine features.

-   Fourier Order ($N$): Controls the "smoothness" of the seasonality. A low $N$ acts as a low-pass filter, capturing only general patterns. A high $N$ allows capturing rapid, high-frequency changes in the cycle.

-   Regularization (Ridge): The coefficients $a_n$ and $b_n$ are estimated by regressing the data onto this feature matrix. Crucially, a normal prior $\mathcal{N}(0, \sigma^2)$ is imposed on these coefficients. This is equivalent to L2 Regularization (Ridge), preventing the seasonality curve from oscillating violently to fit noise. The hyperparameter `seasonality_prior_scale` controls this variance $\sigma^2$.

#### 4.1.3. Holidays and Events ($h(t)$)

Business time series often exhibit sharp, predictable spikes or drops surrounding specific events (e.g., "Black Friday", "Christmas", or "New Year"). Unlike smooth seasonality, these events occur irregularly or follow a lunar calendar (like Easter), making them difficult for standard Fourier series to capture.

Prophet models the holiday component $h(t)$ as a set of independent indicator functions. Mathematically, it constructs a matrix of regressors $Z(t)$:

$$
h(t) = Z(t)\boldsymbol{\kappa}
$$

Where:

-   $Z(t)$: Is a matrix of zeros and ones. Each column represents a specific holiday. $Z_{i,j} = 1$ if time $t_i$ corresponds to holiday $j$, and 0 otherwise.

-   $\boldsymbol{\kappa} \sim \mathcal{N}(0, \nu^2)$: Is the vector of change parameters (the "impact" of each holiday).

**Regularization:**

Similar to seasonality, Prophet applies a Normal prior to the holiday coefficients $\boldsymbol{\kappa}$. This acts as L2 Regularization (Ridge). The parameter $\nu$ (controlled by `holidays_prior_scale`) determines how much the holiday is allowed to alter the forecast:

-   A low scale dampens the holiday effect (assuming it has little impact).

-   A high scale allows for massive spikes in the forecast.

**Built-in Country Holidays:**

A distinctive feature *of Prophet is its internal database of national holidays. Instead of requiring the analyst to manually construct the binary matrix* $Z(t)$ *(a tedious and error-prone process), Prophet allows for automatic injection of country-specific holidays*. This method automatically handles fixed dates and moving dates, ensuring the $h(t)$ component is accurately aligned with the calendar.

#### 4.1.4. Uncertainty Quantification

Unlike deterministic models, Prophet provides uncertainty intervals based on a generative approach.

While the observation noise ($\epsilon_t$) is assumed constant, the main source of risk in time series is trend uncertainty (the risk that the growth rate will change in the future). To estimate this, the model assumes that the average frequency and magnitude of trend changes in the future ($\delta$) will follow the same statistical distribution observed in the past.

Through Monte Carlo sampling, the model simulates multiple possible future trajectories of the trend by sampling new $\delta_j$ from the fitted Laplace distribution. The quantiles of these simulations (typically the 10th and 90th percentiles) form the prediction intervals.

### 4.2. Implementation with Prophet

We will apply Prophet to the Australian Drug Sales dataset (`h2o`). Unlike `skforecast`, which requires the index to be the date, Prophet requires a specific DataFrame format with two columns:

-   `ds` (datestamp): The date column.

-   `y` (numeric): The target measurement.

In [ ]:
#| label: prophet-implementation
#| echo: true

import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from skforecast.datasets import fetch_dataset
from sklearn.metrics import mean_absolute_error

# 1. Load Data
# We reuse the h2o dataset but formatted for Prophet
data = fetch_dataset(name='h2o', raw=True)
data['fecha'] = pd.to_datetime(data['fecha'], format='%Y-%m-%d')

# Rename columns strictly to 'ds' and 'y'
prophet_df = data.rename(columns={'fecha': 'ds', 'x': 'y'})

# 2. Split Data (Keep last 36 months for testing)
train_size = len(prophet_df) - 36
prophet_train = prophet_df.iloc[:train_size]
prophet_test  = prophet_df.iloc[train_size:]

# 3. Initialize and Fit
# We use multiplicative seasonality because drug sales variance grows with the trend
prophet_model = Prophet(
    yearly_seasonality=True,    
    weekly_seasonality=False,   # Monthly data doesn't have weekly cycles
    daily_seasonality=False,   
    interval_width=0.80,  # Default confidence level for the prediction interval
    seasonality_mode='multiplicative', 
    changepoint_prior_scale=0.05, # Default L1 regularization (Flexibility)
    seasonality_prior_scale=10.0  # Default L2 regularization (Seasonality)
)

prophet_model.fit(prophet_train)

# 4. Predict
# Prophet needs a dataframe with future dates to predict
future = prophet_model.make_future_dataframe(periods=36, freq='MS')
forecast = prophet_model.predict(future)

# 5. Evaluate
# Extract predictions corresponding to the test set
pred_prophet = forecast.set_index('ds').loc[prophet_test['ds']]['yhat']
mae_prophet = mean_absolute_error(prophet_test['y'], pred_prophet)

print(f"Prophet MAE: {mae_prophet:.4f}")

# 6. Visualization
fig, ax = plt.subplots(figsize=(12, 6))
# Plot Training Data
ax.plot(prophet_train['ds'], prophet_train['y'], label='Training Data', color='gray', alpha=0.5)
# Plot Actual Test Data
ax.plot(prophet_test['ds'], prophet_test['y'], label='Actual Test Data', color='black')
# Plot Forecast
ax.plot(prophet_test['ds'], pred_prophet, label='Prophet Forecast', color='blue')

# Plot Uncertainty Intervals (The "Blue Cone")
test_forecast = forecast.set_index('ds').loc[prophet_test['ds']]
ax.fill_between(
    test_forecast.index,
    test_forecast['yhat_lower'],
    test_forecast['yhat_upper'],
    color='blue', alpha=0.2, label='Prediction interval'
)

ax.set_title('Prophet Forecast: Australian Drug Sales')
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()

#### 4.2.1. Visualizing Components

One of Prophet's strongest features is its built-in visualization, which breaks down the forecast into its additive components. This allows for "Explainable AI" in a forecasting context.

\# Plot the decomposition (Trend + Seasonality)

fig2 = prophet_model.plot_components(forecast)

plt.show()

In [ ]:
#| label: prophet-components
#| echo: true

# Plot the decomposition (Trend + Seasonality)
fig2 = prophet_model.plot_components(forecast)
plt.show()

**Interpretation of the Components:**

The decomposition plot provides crucial insights into the internal mechanics of the model and validates our hyperparameter choices:

1.  Trend Component (Top Panel):

    The model captures a robust, long-term positive growth in drug sales, rising from approximately 0.4 to over 1.1 index units between 1992 and 2008.

    -   Changepoints: The trend line is not perfectly linear; it exhibits slight inflections (e.g., a subtle softening of the slope around 1998). This confirms that the L1 Regularization (controlled by `changepoint_prior_scale=0.05`) allowed the model to adapt to structural changes in the growth rate without overfitting the noise.

    -   Uncertainty: The light blue shading represents the uncertainty interval. It naturally widens towards the end (2008), reflecting the increasing risk associated with projecting the trend further into the future.

2.  Yearly Seasonality (Bottom Panel):

    This panel validates the use of `seasonality_mode='multiplicative'`.

    -   Percentage Scale: Notice that the Y-axis is labeled in percentages (e.g., $+40\%$, $-60\%$). This means the seasonality is not adding a fixed number of pills sold, but rather multiplying the current trend by a factor.

    -   Seasonal Pattern: The data shows a massive seasonal variation. Sales drop significantly in March (nearly $60-80\%$ below the trend) and spike during the Australian winter (July/August) and again at the end of the year (December/January).

    -   Magnitude: The fact that the seasonality swings from $-80\%$ to $+50\%$ confirms why a linear (additive) model would fail: the fluctuations are too large to be constant constants; they scale proportionally with the growing market size.

#### 4.2.2. Including holidays

In practice, Prophet abstracts the complexity of constructing the regressor matrix $Z(t)$ to include holidays. The library offers two distinct methods to inject external events into the model:

1.  **Built-in National Holidays:** For standard calendar events, we use the `add_country_holidays` method. This requires the ISO code of the country (e.g., 'US', 'AU', 'ES').

2.  **Custom Holidays:** For business-specific events (e.g., "Product Launch", "Marketing Campaign"), we must construct a pandas DataFrame and pass it to the model constructor.

The following example demonstrates the syntax for both approaches:

In [ ]:
#| label: holiday-syntax-example
#| eval: false
#| echo: true

# 1. Load Data
# We reuse the h2o dataset but formatted for Prophet
data = fetch_dataset(name='h2o', raw=True)
data['fecha'] = pd.to_datetime(data['fecha'], format='%Y-%m-%d')

# Rename columns strictly to 'ds' and 'y'
prophet_df = data.rename(columns={'fecha': 'ds', 'x': 'y'})

# 2. Split Data (Keep last 36 months for testing)
train_size = len(prophet_df) - 36
prophet_train = prophet_df.iloc[:train_size]
prophet_test  = prophet_df.iloc[train_size:]

# --- Option A: Built-in National Holidays (Recommended for General Events) ---
# 1. Initialize the model
m = Prophet(seasonality_mode='multiplicative') # the rest of the parameters takes default values (the same as above example)

# 2. Add holidays by country ISO code (e.g., 'AU' for Australia)
# This automatically generates the regressors for Christmas, Easter, etc.
m.add_country_holidays(country_name='AU')

# 3. Fit the model
m.fit(prophet_train)

# Check which holidays were included
print(m.train_holiday_names)

# 4. Predict
# Prophet needs a dataframe with future dates to predict
future = m.make_future_dataframe(periods=36, freq='MS')
forecast = m.predict(future)

# 5. Evaluate
# Extract predictions corresponding to the test set
pred_prophet = forecast.set_index('ds').loc[prophet_test['ds']]['yhat']
mae_prophet = mean_absolute_error(prophet_test['y'], pred_prophet)

print(f"Prophet MAE: {mae_prophet:.4f}")


# 6. Visualization
fig, ax = plt.subplots(figsize=(12, 6))
# Plot Training Data
ax.plot(prophet_train['ds'], prophet_train['y'], label='Training Data', color='gray', alpha=0.5)
# Plot Actual Test Data
ax.plot(prophet_test['ds'], prophet_test['y'], label='Actual Test Data', color='black')
# Plot Forecast
ax.plot(prophet_test['ds'], pred_prophet, label='Prophet Forecast', color='blue')

# Plot Uncertainty Intervals (The "Blue Cone")
test_forecast = forecast.set_index('ds').loc[prophet_test['ds']]
ax.fill_between(
    test_forecast.index,
    test_forecast['yhat_lower'],
    test_forecast['yhat_upper'],
    color='blue', alpha=0.2, label='Prediction interval'
)

ax.set_title('Prophet Forecast with automatic holidays: Australian Drug Sales')
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 7. Plot the decomposition (Trend + Seasonality)
fig2 = m.plot_components(forecast)
plt.show()

In [ ]:
# --- Option B: Custom Business Events (Manual) ---
# 1. Define the event dates and their window of influence
promo_events = pd.DataFrame({
    'holiday': 'super_promo',
    'ds': pd.to_datetime(['2020-06-01', '2021-06-01']),
    'lower_window': 0,  # Days of effect before the date
    'upper_window': 1   # Days of effect after the date
})

# 2. Pass the dataframe with holidays to the constructor
m_custom = Prophet(holidays=promo_events)


# 3. Fit the model
m_custom.fit(prophet_train)

# Check which holidays were included
print(m_custom.train_holiday_names)

# 4. Predict
# Prophet needs a dataframe with future dates to predict
future = m_custom.make_future_dataframe(periods=36, freq='MS')
forecast = m_custom.predict(future)

# 5. Evaluate
# Extract predictions corresponding to the test set
pred_prophet = forecast.set_index('ds').loc[prophet_test['ds']]['yhat']
mae_prophet = mean_absolute_error(prophet_test['y'], pred_prophet)

print(f"Prophet MAE: {mae_prophet:.4f}")


# 6. Visualization
fig, ax = plt.subplots(figsize=(12, 6))
# Plot Training Data
ax.plot(prophet_train['ds'], prophet_train['y'], label='Training Data', color='gray', alpha=0.5)
# Plot Actual Test Data
ax.plot(prophet_test['ds'], prophet_test['y'], label='Actual Test Data', color='black')
# Plot Forecast
ax.plot(prophet_test['ds'], pred_prophet, label='Prophet Forecast', color='blue')

# Plot Uncertainty Intervals (The "Blue Cone")
test_forecast = forecast.set_index('ds').loc[prophet_test['ds']]
ax.fill_between(
    test_forecast.index,
    test_forecast['yhat_lower'],
    test_forecast['yhat_upper'],
    color='blue', alpha=0.2, label='Prediction interval'
)

ax.set_title('Prophet Forecast with automatic holidays: Australian Drug Sales')
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 7. Plot the decomposition (Trend + Seasonality)
fig2 = m_custom.plot_components(forecast)
plt.show()

#### 4.2.3. Hyperparameter Tuning

While Prophet works well out-of-the-box, fine-tuning its parameters can significantly improve accuracy. Unlike Scikit-Learn or `skforecast`, the Prophet library does not include a built-in grid search method. The standard approach recommended by its developers is to iterate over a `ParameterGrid` and evaluate the model on a validation set.

**Key parameters to tune:**

-   **`changepoint_prior_scale`** (Typical Search Range in log-scale: $0.001 - 0.5$):

    -   $0.001$: Imposes strong L1 regularization, forcing high sparsity on the changepoints. This results in a highly rigid, near-linear trend, suitable for stable time series where structural changes are unlikely..

    -   $0.05$: The default value. Provides a balanced baseline between flexibility and regularization.

    -   $0.5$: Allows for a highly flexible, reactive trend. Useful for capturing rapid structural shifts, but carries a significant risk of overfitting to recent noise.

-   **`seasonality_prior_scale`** (Typical Search Range in log-scale: $0.01 - 10$):

    -   $0.01$: Imposes strong L2 regularization, significantly dampening the magnitude of the seasonality. Effectively suppresses the seasonal component if the signal is weak.

    -   $10$: The default value. Applies minimal regularization, allowing the model to fit seasonal patterns of any magnitude based solely on the data evidence. It is rarely necessary to increase this value further.

-   **`seasonality_mode`**:

    -   `'additive'`: Assumes the seasonal amplitude is constant.

    -   `'multiplicative'`: Assumes the seasonal amplitude grows proportionally with the trend.

**Grid Search Implementation:**

In [ ]:
#| label: prophet-tuning
#| echo: true

from sklearn.model_selection import ParameterGrid
import numpy as np

# 1. Define Grid
param_grid = {
    'changepoint_prior_scale': [0.001, 0.05, 0.1, 0.5],
    'seasonality_prior_scale': [0.01, 1.0, 10.0],
    'seasonality_mode': ['additive', 'multiplicative']
}

best_mae = float('inf')
best_params = {}

print("Tuning Prophet...")

# 2. Iterate
for params in ParameterGrid(param_grid):
    # Initialize model with current params
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        **params
    )
    m.add_country_holidays(country_name='AU')
    m.fit(prophet_train)
    
    # Predict
    future = m.make_future_dataframe(periods=36, freq='MS')
    forecast = m.predict(future)
    
    # Evaluate
    pred = forecast.set_index('ds').loc[prophet_test['ds']]['yhat']
    mae = mean_absolute_error(prophet_test['y'], pred)
    
    if mae < best_mae:
        best_mae = mae
        best_params = params

print(f"\nBest MAE: {best_mae:.4f}")
print(f"Best Parameters: {best_params}")

## Set the best model
final_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    **best_params  # this uses the best_params values
)

# 2. Add Holidays (Crucial: Don't forget this if you used it before!)
final_model.add_country_holidays(country_name='AU')

# 3. Fit on the Training Data
final_model.fit(prophet_train)

# 4. Create Future Dataframe & Predict
future_final = final_model.make_future_dataframe(periods=36, freq='MS')
forecast_final = final_model.predict(future_final)

# 5. Extract predictions for metrics
pred_final = forecast_final.set_index('ds').loc[prophet_test['ds']]['yhat']

# 6. Visualization
fig, ax = plt.subplots(figsize=(12, 6))
# Plot Training Data
ax.plot(prophet_train['ds'], prophet_train['y'], label='Training Data', color='gray', alpha=0.5)
# Plot Actual Test Data
ax.plot(prophet_test['ds'], prophet_test['y'], label='Actual Test Data', color='black')
# Plot Forecast
ax.plot(prophet_test['ds'], pred_final, label='Prophet Forecast', color='blue')

# Plot Uncertainty Intervals (The "Blue Cone")
test_forecast = forecast.set_index('ds').loc[prophet_test['ds']]
ax.fill_between(
    test_forecast.index,
    test_forecast['yhat_lower'],
    test_forecast['yhat_upper'],
    color='blue', alpha=0.2, label='Prediction interval'
)

ax.set_title('Prophet Forecast (best model) with automatic holidays: Australian Drug Sales')
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 7. Plot the decomposition (Trend + Seasonality)
fig2 = final_model.plot_components(forecast_final)
plt.show()

**Interpretation of the Best Configuration**

The grid search yielded a winning configuration (`MAE $\approx$ 0.0829`) characterized by high trend flexibility and strong confidence in pure seasonal patterns. The hyperparameter tuning process yielded the following winning configuration:

-   **`changepoint_prior_scale`:** 0.5 (High flexibility)

-   **`seasonality_mode`:** 'additive'

-   **`seasonality_prior_scale`:** 1.0 (Moderate regularization)

1.  The "Additive Paradox": Statistical Fit vs. Structural Logic

    Visual inspection of the raw time series strongly suggests a multiplicative structure (variance increases with the trend). However, the grid search selected an additive model. This phenomenon occurs because, within the specific window of the test set, the additive model achieved a marginally lower MAE by "locally approximating" the multiplicative behavior.

    While a multiplicative model scales the seasonal wave percentage-wise, the additive model mimics this growth by combining a steep trend with fixed-magnitude seasonality. For a forecast horizon of 36 months, this approximation was mathematically more precise than the multiplicative estimation, even if conceptually less rigorous for long-term projections.

2.  Trend Analysis: "Silent" Flexibility

    A `changepoint_prior_scale` of 0.5 typically risks generating a "nervous" or overfitting trend. However, the resulting trend component (top panel) appears remarkably smooth.

    1.  Interpretation: The high prior scale grants the model *permission* to be flexible, but the Bayesian nature of Prophet only utilizes this flexibility if the data demands it. Since the drug sales data is structurally robust with little random noise, the model did not need to introduce chaotic wiggles.

    2.  The Structural Shift: Notice the slight inflection point (elbow) around 1998-2000. The trend slope increases slightly. By tilting the trend upwards during the latter half of the series, the model lifts the baseline. This allows the *fixed* additive seasonality (approx. $\pm 0.6$ units) to reach the higher peaks of 2008 without needing a multiplicative factor.

3.  Seasonality and Holidays

    1.  Scale: The yearly seasonality (bottom panel) now displays absolute units (ranging from $-0.7$ to $+0.5$) rather than percentages. The shape remains identical to the multiplicative version, confirming that the underlying pattern is stable.

    2.  Holidays (Middle Panel): The inclusion of the holiday component (vertical spikes) captures specific events that add approx. 0.2 units to sales. Isolating these spikes helps the yearly seasonality remain smooth and focused on the cyclical pattern rather than outliers.

Conclusion:

The optimization process favored a high-flexibility additive model. It succeeded by leveraging the flexible trend to adjust the slope perfectly, allowing a constant seasonal pattern to fit the data accurately. While counter-intuitive compared to the visual "multiplicative" expectation, this model offers the best empirical accuracy for the specific test period.

### Exercise 4.8: Trend Flexibility

Prophet's `changepoint_prior_scale` is the most critical parameter for capturing trend shifts.

Fit two Prophet models to the US Retail Employment dataset (`us_retail_employment.xlsx`):

1.  Model A: `changepoint_prior_scale = 0.001` (Very rigid).

2.  Model B: `changepoint_prior_scale = 0.5` (Very flexible).

Compare their plots visually. Does Model A fail to capture the 2008 financial crisis drop? Does Model B react too violently to noise?

## 5. Hybrid Approaches: Combining Statistical and ML Methods

### 5.1 The Hybrid Philosophy

Throughout this course, we have seen two distinct paradigms. Classical statistical models, such as ARIMA, are highly effective at capturing linear patterns, rigid seasonality, and temporal autocorrelation, even when working with limited datasets. On the other hand, Machine Learning models (like XGBoost or Neural Networks) demonstrate a strong capacity for identifying complex non-linear relationships and interactions among exogenous variables. However, they often encounter difficulties to extrapolate simple trends or capture subtle autoregressive patterns without extensive feature engineering.

-   Hybrid approaches aim to combine the "best of both worlds." The underlying hypothesis is that a time series can be decomposed into a linear component (predictable by statistics) and a non-linear component (predictable by ML).

    Common hybrid strategies include:

    -   **Residual Modeling:** First, fit a statistical model to capture the "easy" linear structure. Then, train an ML model to predict the *residuals* (errors) of that statistical model.

-   **Ensemble Averaging:** Train both types of models independently and combine their predictions using a weighted average.

-   **Feature Augmentation:** Use the forecast of a statistical model as just another input feature ($X$) for the Machine Learning model.

### 5.2 Residual Modeling Approach

This strategy is conceptually similar to Gradient Boosting. We treat the time series $y_t$ as a sum of a linear signal and a non-linear residual:

$$
y_t = \underbrace{L_t}_{\text{ARIMA}} + \underbrace{N_t}_{\text{ML}} + \epsilon_t
$$

The steps to follow in the residual modeling approach are:

1.  Train a statistical modelo such as ARIMA on the raw series $y_t$, and calculate the residuals: $r_t = y_t - \hat{y}^{ARIMA}_t$.

2.  Train an ML model (e.g., XGBoost) using the feature matrix $X_t$ to predict these residuals $r_t$.

3.  Forecast: The final prediction is the sum of the ARIMA forecast and the ML correction.

$$
\hat{y}_{t+h} = \hat{y}_{t+h}^{ARIMA} + \hat{r}_{t+h}^{XGBoost}
$$

#### **Step 1: Fit the Statistical Baseline**

We can use `statsmodels` to fit a standard ARIMA/SARIMA model. Ideally, the order $(p,d,q)$ should be determined by the analysis presented in Chapter 3. Alternatively, we can use the `auto_arima` method from `pmdarima`.

**Example with `statsmodels`**

In [ ]:
#| label: hybrid-residual-arima statsmodels
#| echo: true

import pandas as pd
import matplotlib.pyplot as plt
from skforecast.datasets import fetch_dataset
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error

# -----------------------------------------------------------------------------
# 1. Load and Prepare Data
# -----------------------------------------------------------------------------
# Load the Australian Drug Sales dataset
data = fetch_dataset(name='h2o', raw=True)
data['fecha'] = pd.to_datetime(data['fecha'], format='%Y-%m-%d')
data = data.set_index('fecha')
data = data.asfreq('MS') # Set frequency to Month Start
data = data.rename(columns={'x': 'sales'})

# Split Train/Test (Keep last 24 months for testing, consistent with previous sections)
steps = 24
y_train = data['sales'][:-steps]
y_test  = data['sales'][-steps:]

# -----------------------------------------------------------------------------
# 2. Fit Statistical Baseline (ARIMA)
# -----------------------------------------------------------------------------
# We use the parameters identified in our previous analysis (Chapter 3)
# Order: (p, d, q) x (P, D, Q, s)
arima_model = ARIMA(y_train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
arima_fit = arima_model.fit()

# Extract linear signal and residuals (in-sample)
arima_fitted = arima_fit.fittedvalues
arima_residuals = arima_fit.resid

print(f"ARIMA AIC: {arima_fit.aic:.2f}")

# Visualize the residuals to confirm they are stationary but perhaps non-random
plt.figure(figsize=(10, 4))
plt.plot(arima_residuals,'-r')
plt.axhline(0, color='black', linestyle='--', linewidth=0.8) # Reference line
plt.title("ARIMA Residuals (Target for ML Model)")
plt.show()

**example with `pmdarima`**

In [ ]:
#| label: hybrid-residual-autoarima
#| echo: true

import pandas as pd
import matplotlib.pyplot as plt
from skforecast.datasets import fetch_dataset
import pmdarima as pm  
from sklearn.metrics import mean_absolute_error

# -----------------------------------------------------------------------------
# 1. Load and Prepare Data
# -----------------------------------------------------------------------------
# Load the Australian Drug Sales dataset
data = fetch_dataset(name='h2o', raw=True)
data['fecha'] = pd.to_datetime(data['fecha'], format='%Y-%m-%d')
data = data.set_index('fecha')
data = data.asfreq('MS') # Set frequency to Month Start
data = data.rename(columns={'x': 'sales'})

# Split Train/Test (Keep last 24 months for testing)
steps = 24
y_train = data['sales'][:-steps]
y_test  = data['sales'][-steps:]

# -----------------------------------------------------------------------------
# 2. Fit Statistical Baseline (AutoARIMA)
# -----------------------------------------------------------------------------
# Instead of manually specifying parameters, we let AutoARIMA find the best model
# based on minimizing AIC.
print("Searching for optimal ARIMA parameters...")

autoarima_model = pm.auto_arima(
    y_train,
    seasonal=True,
    m=12,               # Frequency of the seasonality (12 for monthly data)
    stepwise=True,      # Uses the stepwise algorithm (faster than full grid)
    suppress_warnings=True,
    error_action='ignore',
    trace=True          # Print the progress of the search
)

print(f"\nBest Model: {autoarima_model.order} x {autoarima_model.seasonal_order}")
print(f"AIC: {autoarima_model.aic():.2f}")

# Extract residuals
autoarima_residuals = autoarima_model.resid()

# Extract fitted values
autoarima_fitted = autoarima_model.fittedvalues()

# Visualize the residuals to confirm they are stationary
plt.figure(figsize=(10, 4))
plt.plot(autoarima_residuals,'-r')
plt.axhline(0, color='black', linestyle='--', linewidth=0.8) # Reference line
plt.title(f"AutoARIMA Residuals (Order: {autoarima_model.order}x{autoarima_model.seasonal_order})")
plt.grid(True, alpha=0.3)
plt.show()

#### Step 2: Modeling the Non-Linear Residuals

Now we treat the ARIMA residuals as our new target variable. The ML model will attempt to find patterns in these errors that ARIMA failed to capture (e.g., non-linear interactions with exogenous variables).

We first generate the feature matrix ($X$) based on the training data. Note that creating lags will introduce NaNs at the beginning of the series, so we must align the indices carefully. In the following example, we will proceed using the residuals from the `statsmodels` implementation, but the workflow is identical if the `auto_arima` model (see Step 1) is chosen.

In [ ]:
#| label: hybrid-train-ml
#| echo: true

from xgboost import XGBRegressor
import pandas as pd

# -----------------------------------------------------------------------------
# 1. Feature Engineering for the ML Model
# -----------------------------------------------------------------------------
# We create a feature matrix (X) to give context to the ML model.
# It needs to know "what was happening" in the series when ARIMA failed.

X_features = pd.DataFrame(index=data['sales'].index)

# A. Lag features (Autoregressive context)
lags = [1, 2, 3, 4, 5, 6, 12]
for lag in lags:
    X_features[f'lag_{lag}'] = data['sales'].shift(lag)

# B. Rolling Window features (Trend/Volatility context)
# Important: Shift by 1 to avoid data leakage (using only past data)
X_features['rolling_mean_6'] = data['sales'].shift(1).rolling(window=6).mean()
X_features['rolling_mean_12'] = data['sales'].shift(1).rolling(window=12).mean()

# C. Calendar features (Seasonality)
X_features['month'] = X_features.index.month

# D. Split into train and test
X_train_features = X_features.iloc[:-steps]
X_test_features = X_features.iloc[-steps:]


# D. Drop NaNs created by lags/rolling in the train set (the test set shoudn't have NaNs values)
X_train_features = X_train_features.dropna()


# -----------------------------------------------------------------------------
# 2. Data Alignment
# -----------------------------------------------------------------------------
# The features (X) now have fewer rows than the residuals because we dropped NaNs.
# We must align them to ensure we are training on the exact same dates.
common_idx = X_train_features.index.intersection(arima_residuals.index)

# Final Training Sets
X_residual_train = X_train_features.loc[common_idx]
y_residual_target = arima_residuals.loc[common_idx]

print(f"Training ML model on {len(X_residual_train)} residuals.")

# -----------------------------------------------------------------------------
# 3. Train XGBoost on Residuals
# -----------------------------------------------------------------------------
# We use a relatively simple tree (max_depth=4) to avoid overfitting the noise
residual_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

residual_model.fit(X_residual_train, y_residual_target)

print("Step 2: Residual ML model trained.")
r2_score = residual_model.score(X_residual_train, y_residual_target)
print(f"Variance of residuals explained by XGBoost (R²): {r2_score:.4f}")

#### Step 3: Generating the Hybrid Forecast

To forecast, we predict both components separately and sum them up.

In [ ]:
#| label: hybrid-forecast-gen
#| echo: true

from sklearn.metrics import mean_absolute_error

# 1. Generate Base Forecast (Linear)
# statsmodels requires the number of steps
arima_forecast = arima_fit.forecast(steps=len(y_test))
# Ensure index match for easy plotting
arima_forecast.index = y_test.index

# 2. Generate Residual Forecast (Non-Linear Correction)
# We use the test set features we prepared in previous sections
ml_correction = pd.Series(residual_model.predict(X_test_features), index=y_test.index)

# 3. Combine
hybrid_forecast = arima_forecast + ml_correction

# 4. Evaluation
print("\n" + "="*40)
print("HYBRID MODEL EVALUATION")
print("="*40)
mae_arima = mean_absolute_error(y_test, arima_forecast)
mae_xgb = mean_absolute_error(y_test-arima_forecast, ml_correction) # MAE with respecto to the residuals in test set
mae_hyb = mean_absolute_error(y_test, hybrid_forecast)
print(f"Baseline (ARIMA only) MAE: {mae_arima:.4f}")
print(f"XGBoost Component MAE: {mae_xgb:.4f}")
print(f"Hybrid (ARIMA + XGBoost) MAE: {mae_hyb:.4f}")

Contrary to initial expectations, the addition of the XGBoost component resulted in an increased MAE compared to the standalone ARIMA model. This indicates that the residual modeling approach is degrading the forecasting ability in this specific case, despite the Machine Learning model showing a high $R^2$ score during the training phase.

##### Exercise 4.9

Analyze the discrepancy between the training performance (Step 2) and the testing performance (Step 3). The XGBoost model explained a large portion of the variance in the training residuals ($R^2 \approx 0.84$). However, it failed to improve the forecast on unseen data.

What statistical phenomenon is likely occurring here? Why might a complex model perform worse than a simple one? Can you propose any improvement in the models to correct this situation?

#### Visualization

Visualizing the components helps us verify if the ML model is actually contributing useful corrections or just modeling noise.

In [ ]:
#| label: hybrid-visualization
#| echo: true

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Plot 1: The Final Forecasts
axes[0].plot(y_test.index, y_test.values, 'k-', label='Actual Sales', linewidth=2)
axes[0].plot(y_test.index, arima_forecast.values, 'b--', label='ARIMA (Linear)', alpha=0.6)
axes[0].plot(y_test.index, hybrid_forecast.values, 'r-', label='Hybrid (Corrected)', linewidth=1.5)
axes[0].set_title('Forecast Comparison: Can the Hybrid model improve the baseline?')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: What did the ML model learn?
# We compare the actual ARIMA errors vs. what XGBoost predicted they would be
axes[1].plot(X_residual_train.index, y_residual_target, 'b-', label='Actual ARIMA Residuals', alpha=0.3)
axes[1].plot(X_residual_train.index, residual_model.predict(X_residual_train), 'r--', label='ML Pattern', linewidth=1)
axes[1].set_title('Residual Analysis: Did ML find a pattern in the errors?')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.3 Ensemble Averaging

Unlike the serial nature of residual modeling—where models build upon one another to correct errors—Ensemble Averaging is a parallel process. It involves the independent development of multiple diverse models, followed by the combination of their respective outputs to enhance predictive stability.

The *Wisdom of Crowds* theory suggests that if the models are uncorrelated (i.e., they make different types of errors), their combined average will be more robust and often more accurate than any single individual model.

There are two main ways to combine forecasts:

1.  Simple Average: Give equal weight to all models. It is a very tough benchmark to beat because it is less prone to overfitting than complex weighting schemes.

    $$
    \hat{y}_{ensemble} = \frac{1}{N} \sum_{i=1}^{N} \hat{y}_i
    $$

2.  Weighted Average (Inverse Error): Assign higher voting power to models with lower error in the validation set. The most popular metric to obtain ensemble weights in machine learning approaches is the Mean Squared Error (MSE). However any other metric can be used:

    $$
    W_i = \frac{1}{\text{MSE}_i}
    $$

#### 5.3.1. Implementation in Python

We will combine the predictions from previous sections.

In a production scenario, the weights for the weighted average must be calculated on a separate Validation Set to avoid Data Leakage. For this demonstration, we will show the Simple Average approach, which is robust and leakage-free.

In [ ]:
#| label: ensemble-full-retrain
#| echo: true

import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

# -----------------------------------------------------------------------------
# 1. Re-Train Candidate Models (Independent Experts)
# -----------------------------------------------------------------------------

# --- Model A: ARIMA (Statistical Expert) ---
# We use the full y_train history
print("Training Model A: ARIMA...")
arima_model_ens = ARIMA(y_train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
arima_fit_ens = arima_model_ens.fit()

# Forecast
pred_arima = arima_fit_ens.forecast(steps=len(y_test))
pred_arima.index = y_test.index

# --- Model B: Pure XGBoost (Machine Learning Expert) ---
# We use the feature matrix X created in Section 5.2.
# CRITICAL: We must align y_train to X_train_features (removing the first 12 NaNs)
common_idx = X_train_features.index.intersection(y_train.index)

X_train_ml = X_train_features.loc[common_idx]
y_train_ml = y_train.loc[common_idx] # Target is SALES, not residuals

print(f"Training Model B: Pure XGBoost on {len(X_train_ml)} samples...")
xgb_model_ens = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    # Anti-Redundancy Strategy:
    reg_alpha=1.0,        # Moderate L1 (Lasso) to remove redundant features
    reg_lambda=1.0,       # Standard L2 (Ridge) for stability
    colsample_bytree=0.7, # Use only 70% of features per tree (decorrelates trees)
    learning_rate=0.05,
    random_state=42
)
xgb_model_ens.fit(X_train_ml, y_train_ml)

# Forecast
pred_xgb_pure = pd.Series(xgb_model_ens.predict(X_test_features), index=y_test.index)

# -----------------------------------------------------------------------------
# 2. Generate Ensemble Forecast (Simple Average)
# -----------------------------------------------------------------------------
# We create a DataFrame to hold the diverse opinions
ensemble_df = pd.DataFrame(index=y_test.index)
ensemble_df['ARIMA'] = pred_arima
ensemble_df['XGBoost'] = pred_xgb_pure

# The Consensus: Simple Mean of the models
ensemble_df['Ensemble_Avg'] = ensemble_df.mean(axis=1)

# -----------------------------------------------------------------------------
# 3. Evaluation
# -----------------------------------------------------------------------------
def evaluate_forecast(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    print(f"{name.ljust(25)} MAE: {mae:.4f}")

print("\n" + "="*40)
print("ENSEMBLE VS INDIVIDUAL MODELS")
print("="*40)

evaluate_forecast(y_test, ensemble_df['ARIMA'], "1. ARIMA (Standalone)")
evaluate_forecast(y_test, ensemble_df['XGBoost'], "2. XGBoost (Standalone)")
print("-" * 40)
evaluate_forecast(y_test, ensemble_df['Ensemble_Avg'], "3. Simple Ensemble (Mean)")

##### Exercise 4.10

Use a Train+Validation+Test schema and implement a weighted averaged ensemble model using the same data set. Use the MSE metric to evaluate the error on the validation set and use it to weigt the final forecast of an ensemble of ARIMA, XGBoost and Prophet models.

#### 5.3.2. Advanced Model Stacking

Stacking is an advanced ensemble technique. Instead of using a simple formula (like average) to combine predictions, we use a machine learning model (the Meta-Learner) to learn *how* to combine them best.

##### Exercise 4.11

The goal is to verify if a Meta-Learner can find a better combination than a simple average. Use the `h2o` dataset and follow the steps given below:

1.  Data Preparation: Create a new DataFrame `X_stack` where the columns are the predictions of your base models (ARIMA, XGBoost, Prophet, etc.) and the target `y` is the actual sales from the Test set.

2.  Constraint (Data Leakage Warning): you should use "Out-of-Fold" predictions on the training set. If you use the Test set predictions to train the meta-learner this is data leakage.

3.  Meta-Model Training: Train a `LinearRegression` (from `sklearn.linear_model`) using `X_stack` as features and the actual sales `y_test` as the target.

    $$
    \hat{y}_{final} = \beta_0 + \beta_1 \hat{y}_{ARIMA} + \beta_2 \hat{y}_{XGBoost} + \dots
    $$

4.  Inspect Weights: Check the `.coef_` attribute of the linear regression.

    -   Which model does the Meta-Learner trust the most (highest coefficient)?

    -   Did it assign negative weights to any model? (This often happens in Stacking to correct the bias of another model).

## 6. Final Recomendations

### 6.1 When to Use ML for Time Series

**ML approaches are particularly suited when:**

-   The relationship between features and target is non-linear
-   Multiple exogenous variables are available
-   Domain-specific features can be engineered
-   The series has complex seasonality that's hard to specify parametrically
-   Large amounts of training data are available

**Classical statistical methods may be preferred when:**

-   Data is limited
-   Interpretability of parameters is crucial
-   Uncertainty quantification is important
-   The series follows well-known patterns (pure AR, trend + seasonality)

### 6.2 Best Practices

**Data Preparation:**

-   Ensure chronological ordering (never shuffle time series data!)
-   Check for and handle missing values appropriately
-   Consider differencing or transformation if needed

**Feature Engineering:**

-   Include relevant lag features (guided by ACF/PACF)
-   Add rolling statistics to capture local trends/volatility
-   Include calendar features for seasonal patterns
-   Consider Fourier terms for smooth seasonality
-   Watch for data leakage (no future information in features!)

**Model Training:**

-   Use time-respecting train/test splits
-   Apply appropriate scaling for regularized models and NNs
-   Consider multiple models and compare systematically
-   Use cross-validation with TimeSeriesSplit

**Evaluation:**

-   Use multiple metrics (RMSE, MAE, MAPE, MASE)
-   Visualize predictions against actuals
-   Analyze residuals for remaining patterns
-   Consider prediction intervals, not just point forecasts

## Exercises

### Exercise Set A: Feature Engineering

**A.1** Create a function that automatically selects the optimal number of lag features using cross-validation. Test it on the drug sales dataset.

**A.2** Investigate the impact of different rolling window sizes (3, 6, 12, 24 months) on forecast accuracy for the Random Forest model. Plot the results.

**A.3** Compare the performance of models using: (a) only lag features, (b) only calendar features, (c) only Fourier terms, (d) all features combined. What does this tell you about the structure of the drug sales data?

### Exercise Set B: Model Comparison

**B.1** Implement a grid search for XGBoost hyperparameters using TimeSeriesSplit cross-validation. Report the best parameters and their cross-validated performance.

**B.2** Compare one-step-ahead versus multi-step-ahead (h=6) forecasting for each ML model. Does the relative ranking of models change?

**B.3** Download a different time series (e.g., airline passengers, electricity demand) and compare whether the best model remains the same.

### Exercise Set C: Advanced Topics

**C.1** Implement a **stacking ensemble** that uses the predictions of Ridge, Random Forest, and XGBoost as features for a meta-learner. Compare with simple averaging.

**C.2** Extend the hybrid ARIMA+ML approach by trying different statistical base models (Holt-Winters, SARIMA). Does the choice of base model matter?

**C.3** **Critical Analysis:** Many ML papers report impressive results on time series benchmarks. Design an experiment to test whether a simple ARIMA baseline is actually competitive with the best ML models on your chosen dataset. Document your findings critically.

### Exercise Set D: Prophet Deep Dive

**D.1** Use Prophet's cross-validation functionality (`prophet.diagnostics.cross_validation`) to evaluate forecast accuracy at different horizons. Plot the MAPE by horizon.

**D.2** Experiment with Prophet's `add_regressor` functionality to include exogenous variables. Create a synthetic regressor (e.g., a leading indicator) and assess its impact.

**D.3** Compare Prophet's uncertainty intervals with bootstrap intervals from a Random Forest model. Which seems more calibrated?

------------------------------------------------------------------------

## References

**Core Readings:**

-   Hyndman, R.J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice*, 3rd ed. Chapter 12: Advanced Forecasting Methods. [Online](https://otexts.com/fpp3/)

-   Taylor, S.J., & Letham, B. (2018). Forecasting at scale. *The American Statistician*, 72(1), 37-45.

**Machine Learning for Time Series:**

-   Bontempi, G., Taieb, S.B., & Le Borgne, Y.A. (2013). Machine learning strategies for time series forecasting. *European Business Intelligence Summer School*, 62-77.

-   Montero-Manso, P., & Hyndman, R.J. (2021). Principles and algorithms for forecasting groups of time series: Locality and globality. *International Journal of Forecasting*, 37(4), 1632-1653.

**Practical Resources:**

-   Skforecast documentation: https://skforecast.org/
-   Prophet documentation: https://facebook.github.io/prophet/